<a href="https://colab.research.google.com/github/hammadshub/Hierarchical-Graph-Attention-Networks/blob/main/Hierarchical_Graph_Attention_Networks_for_Long_Context_Document_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from torch_geometric.data import HeteroData
from torch.utils.data import Dataset as TorchDataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import T5Tokenizer, T5ForConditionalGeneration

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Week 1/2
def split_sentences(article_text):
    return [s.strip() for s in article_text.split("\n") if s.strip()]

def build_hierarchy(sentences, sentences_per_paragraph=5, paragraphs_per_section=4):
    paragraphs = [sentences[i:i+sentences_per_paragraph] for i in range(0, len(sentences), sentences_per_paragraph)]
    sections = [paragraphs[i:i+paragraphs_per_section] for i in range(0, len(paragraphs), paragraphs_per_section)]
    return sections

def build_graph(hierarchy, embedding_dim=128):
    data = HeteroData()
    sentence_list, sent_to_para, para_to_section = [], [], []
    para_counter = 0
    for section_idx, section in enumerate(hierarchy):
        for para in section:
            for sent in para:
                sentence_list.append(sent)
                sent_to_para.append(para_counter)
            para_to_section.append(section_idx)
            para_counter += 1
    num_sentences, num_paragraphs, num_sections = len(sentence_list), para_counter, len(hierarchy)
    data['sentence'].x = torch.randn(num_sentences, embedding_dim)
    data['paragraph'].x = torch.randn(num_paragraphs, embedding_dim)
    data['section'].x = torch.randn(num_sections, embedding_dim)
    data['document'].x = torch.randn(1, embedding_dim)
    data['sentence','belongs_to','paragraph'].edge_index = torch.tensor([list(range(num_sentences)), sent_to_para])
    data['paragraph','belongs_to','section'].edge_index = torch.tensor([list(range(num_paragraphs)), para_to_section])
    data['section','belongs_to','document'].edge_index = torch.tensor([list(range(num_sections)), [0]*num_sections])
    return data, sentence_list

def build_semantic_edges(sentence_list, embedder, threshold=0.6, max_edges_per_node=5):
    n = len(sentence_list)
    if n < 2:
        return torch.tensor([[], []], dtype=torch.long), None
    embeddings = embedder.encode(sentence_list)
    sim_matrix = cosine_similarity(embeddings)
    src, dst = [], []
    for i in range(n):
        sims = sim_matrix[i].copy()
        sims[i] = -1
        top_k_idx = np.argsort(sims)[::-1][:max_edges_per_node]
        for j in top_k_idx:
            if sims[j] >= threshold:
                src.append(i); dst.append(j)
    return torch.tensor([src, dst]), sim_matrix

def build_full_graph(article_text, embedder, embedding_dim=384, threshold=0.6, max_edges_per_node=5):
    sentences = split_sentences(article_text)
    hierarchy = build_hierarchy(sentences)
    graph, sentence_list = build_graph(hierarchy, embedding_dim=embedding_dim)
    real_embeddings = embedder.encode(sentence_list)
    graph['sentence'].x = torch.tensor(real_embeddings, dtype=torch.float)
    sem_edge_index, _ = build_semantic_edges(sentence_list, embedder, threshold, max_edges_per_node)
    graph['sentence','similar_to','sentence'].edge_index = sem_edge_index
    return graph

class GraphSummarizationDataset(TorchDataset):
    def __init__(self, hf_dataset, embedder, embedding_dim=384, threshold=0.6, max_edges_per_node=5):
        self.embedder = embedder
        self.embedding_dim = embedding_dim
        self.threshold = threshold
        self.max_edges_per_node = max_edges_per_node
        self.valid_indices = [i for i in range(len(hf_dataset)) if len(hf_dataset[i]["article"].strip()) > 0]
        self.hf_dataset = hf_dataset
    def __len__(self):
        return len(self.valid_indices)
    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        article = self.hf_dataset[real_idx]["article"]
        summary = self.hf_dataset[real_idx]["abstract"]
        graph = build_full_graph(article, self.embedder, self.embedding_dim, self.threshold, self.max_edges_per_node)
        return graph, summary


dataset_50 = load_dataset("ccdv/pubmed-summarization", split="train[:50]")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
graph_dataset = GraphSummarizationDataset(dataset_50, embedder)
print(f"Setup complete. graph_dataset size: {len(graph_dataset)}")

Device: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Setup complete. graph_dataset size: 48


In [ ]:
!pip install torch-geometric transformers datasets evaluate rouge-score bert-score spacy -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.5 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset

dataset = load_dataset("ccdv/pubmed-summarization", split="train[:20]")
print("Loaded", len(dataset), "examples")
print()
print("--- Example document (first 1000 chars) ---")
print(dataset[0]["article"][:1000])
print()
print("--- Example summary ---")
print(dataset[0]["abstract"])

README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

section/train-00000-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  210MB            

section/train-00000-of-00005.parquet: downloading bytes:           |  0.00B            

section/train-00001-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  208MB            

section/train-00001-of-00005.parquet: downloading bytes:           |  0.00B            

...

section/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/119924 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6633 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6658 [00:00<?, ? examples/s]

Loaded 20 examples

--- Example document (first 1000 chars) ---
a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . 
 in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . 
 the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . 
 anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . 
 snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states 
 there are also some reports

In [ ]:
lengths = [len(dataset[i]["article"].split()) for i in range(20)]
summary_lengths = [len(dataset[i]["abstract"].split()) for i in range(20)]
print("Avg document length (words):", sum(lengths)/len(lengths))
print("Avg summary length (words):", sum(summary_lengths)/len(summary_lengths))

Avg document length (words): 2859.6
Avg summary length (words): 219.6


In [ ]:
sample = dataset[0]["article"]
lines = sample.split("\n")
print("Number of newline-separated chunks:", len(lines))
print()
for l in lines[:5]:
    print(repr(l))

Number of newline-separated chunks: 153

'a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . '
' in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . '
' the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . '
' anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . '
' snack should have 300 - 400 kcal energy and could provide 5 - 10 g of protein / day . nowadays , school nutrition programs are running as the national programs , world - wide . national school lunch program in the united states '


In [ ]:
for doc_idx in [1, 5, 10]:
    lines = [l for l in dataset[doc_idx]["article"].split("\n") if l.strip()]
    print(f"Doc {doc_idx}: {len(lines)} lines, first line: {lines[0][:80]}")

Doc 1: 74 lines, first line: it occurs in more than 50% of patients and may reach 90% in certain types of can
Doc 5: 69 lines, first line: world - wide , infertility affects 1015% of couples who are trying to conceive ,
Doc 10: 101 lines, first line: an exponential rise in alzheimer 's disease ( ad ) prevalence rates is predicted


In [ ]:
import torch
from torch_geometric.data import HeteroData

data = HeteroData()

data['sentence'].x = torch.randn(3, 128)
data['section'].x = torch.randn(2, 128)
data['document'].x = torch.randn(1, 128)

data['sentence', 'belongs_to', 'section'].edge_index = torch.tensor([
    [0, 1, 2],
    [0, 0, 1]
])

data['section', 'belongs_to', 'document'].edge_index = torch.tensor([
    [0, 1],
    [0, 0]
])

print("Graph constructed successfully:")
print(data)

Graph constructed successfully:
HeteroData(
  sentence={ x=[3, 128] },
  section={ x=[2, 128] },
  document={ x=[1, 128] },
  (sentence, belongs_to, section)={ edge_index=[2, 3] },
  (section, belongs_to, document)={ edge_index=[2, 2] }
)


# Literature Review — Week 1

## Papers
1. Veličković et al., Graph Attention Networks (2018)
    
2. Heterogeneous graph paper
   
3. Long-document summarization paper
   

## Dataset findings (ccdv/pubmed-summarization)
- Avg document length: 2,859.6 words (~3,700+ tokens estimated)
- Avg summary length: 219.6 words
- T5-small max input: 512 tokens → ~86% of average document gets truncated by a flat baseline
- Sentences are pre-split by newline (\n), confirmed consistent across docs 0, 1, 5, 10 — no NLP sentence-splitter library needed for Week 2

## Environment notes
- Original `scientific_papers` HF dataset is deprecated (script-based loading no longer supported)
- Using `ccdv/pubmed-summarization` instead — same PubMed data, Parquet-based, script-free

In [ ]:
def split_sentences(article_text):

    sentences = [s.strip() for s in article_text.split("\n") if s.strip()]
    return sentences

# Test on 10 documents
for i in range(10):
    sents = split_sentences(dataset[i]["article"])
    print(f"Doc {i}: {len(sents)} sentences")


sents = split_sentences(dataset[0]["article"])
short_sents = [s for s in sents if len(s.split()) < 3]
print(f"Suspiciously short 'sentences' (<3 words): {len(short_sents)}")
print(short_sents[:10])

Doc 0: 153 sentences
Doc 1: 74 sentences
Doc 2: 36 sentences
Doc 3: 175 sentences
Doc 4: 39 sentences
Doc 5: 69 sentences
Doc 6: 39 sentences
Doc 7: 82 sentences
Doc 8: 89 sentences
Doc 9: 27 sentences
Suspiciously short 'sentences' (<3 words): 0
[]


Every 5 sentences → 1 paragraph

---


Every 4 paragraphs → 1 section

---



In [ ]:
def build_hierarchy(sentences, sentences_per_paragraph=5, paragraphs_per_section=4):

    paragraphs = []
    for i in range(0, len(sentences), sentences_per_paragraph):
        paragraphs.append(sentences[i:i + sentences_per_paragraph])

    sections = []
    for i in range(0, len(paragraphs), paragraphs_per_section):
        sections.append(paragraphs[i:i + paragraphs_per_section])

    return sections

sentences = split_sentences(dataset[0]["article"])
hierarchy = build_hierarchy(sentences)

print(f"Doc 0: {len(sentences)} sentences")
print(f"Grouped into {len(hierarchy)} sections")
for s_idx, section in enumerate(hierarchy):
    print(f"  Section {s_idx}: {len(section)} paragraphs")
    for p_idx, para in enumerate(section):
        print(f"    Paragraph {p_idx}: {len(para)} sentences")

Doc 0: 153 sentences
Grouped into 8 sections
  Section 0: 4 paragraphs
    Paragraph 0: 5 sentences
    Paragraph 1: 5 sentences
    Paragraph 2: 5 sentences
    Paragraph 3: 5 sentences
  Section 1: 4 paragraphs
    Paragraph 0: 5 sentences
    Paragraph 1: 5 sentences
    Paragraph 2: 5 sentences
    Paragraph 3: 5 sentences
  Section 2: 4 paragraphs
    Paragraph 0: 5 sentences
    Paragraph 1: 5 sentences
    Paragraph 2: 5 sentences
    Paragraph 3: 5 sentences
  Section 3: 4 paragraphs
    Paragraph 0: 5 sentences
    Paragraph 1: 5 sentences
    Paragraph 2: 5 sentences
    Paragraph 3: 5 sentences
  Section 4: 4 paragraphs
    Paragraph 0: 5 sentences
    Paragraph 1: 5 sentences
    Paragraph 2: 5 sentences
    Paragraph 3: 5 sentences
  Section 5: 4 paragraphs
    Paragraph 0: 5 sentences
    Paragraph 1: 5 sentences
    Paragraph 2: 5 sentences
    Paragraph 3: 5 sentences
  Section 6: 4 paragraphs
    Paragraph 0: 5 sentences
    Paragraph 1: 5 sentences
    Paragraph 2: 5 

In [ ]:
for doc_idx in [9, 3]:
    sentences = split_sentences(dataset[doc_idx]["article"])
    hierarchy = build_hierarchy(sentences)
    print(f"Doc {doc_idx}: {len(sentences)} sentences -> {len(hierarchy)} sections")
    for s_idx, section in enumerate(hierarchy):
        para_sizes = [len(p) for p in section]
        print(f"  Section {s_idx}: {len(section)} paragraphs, sentence counts {para_sizes}")
    print()

Doc 9: 27 sentences -> 2 sections
  Section 0: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 1: 2 paragraphs, sentence counts [5, 2]

Doc 3: 175 sentences -> 9 sections
  Section 0: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 1: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 2: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 3: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 4: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 5: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 6: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 7: 4 paragraphs, sentence counts [5, 5, 5, 5]
  Section 8: 3 paragraphs, sentence counts [5, 5, 5]



In [ ]:
import torch
from torch_geometric.data import HeteroData

def build_graph(hierarchy, embedding_dim=128):

    data = HeteroData()

    sentence_list = []
    sent_to_para = []
    para_to_section = []

    para_counter = 0
    for section_idx, section in enumerate(hierarchy):
        for para in section:
            for sent in para:
                sentence_list.append(sent)
                sent_to_para.append(para_counter)
            para_to_section.append(section_idx)
            para_counter += 1

    num_sentences = len(sentence_list)
    num_paragraphs = para_counter
    num_sections = len(hierarchy)

    data['sentence'].x = torch.randn(num_sentences, embedding_dim)
    data['paragraph'].x = torch.randn(num_paragraphs, embedding_dim)
    data['section'].x = torch.randn(num_sections, embedding_dim)
    data['document'].x = torch.randn(1, embedding_dim)

    sent_src = list(range(num_sentences))
    sent_dst = sent_to_para
    data['sentence', 'belongs_to', 'paragraph'].edge_index = torch.tensor([sent_src, sent_dst])


    para_src = list(range(num_paragraphs))
    para_dst = para_to_section
    data['paragraph', 'belongs_to', 'section'].edge_index = torch.tensor([para_src, para_dst])

    sec_src = list(range(num_sections))
    sec_dst = [0] * num_sections
    data['section', 'belongs_to', 'document'].edge_index = torch.tensor([sec_src, sec_dst])

    return data, sentence_list

sentences = split_sentences(dataset[0]["article"])
hierarchy = build_hierarchy(sentences)
graph, sentence_list = build_graph(hierarchy)

print(graph)
print(f"\nSanity check: {len(sentence_list)} sentences captured, expected {len(sentences)}")

HeteroData(
  sentence={ x=[153, 128] },
  paragraph={ x=[31, 128] },
  section={ x=[8, 128] },
  document={ x=[1, 128] },
  (sentence, belongs_to, paragraph)={ edge_index=[2, 153] },
  (paragraph, belongs_to, section)={ edge_index=[2, 31] },
  (section, belongs_to, document)={ edge_index=[2, 8] }
)

Sanity check: 153 sentences captured, expected 153


In [ ]:
for doc_idx in [9, 3]:
    sentences = split_sentences(dataset[doc_idx]["article"])
    hierarchy = build_hierarchy(sentences)
    graph, sentence_list = build_graph(hierarchy)
    print(f"Doc {doc_idx}:")
    print(graph)
    print(f"Sanity check: {len(sentence_list)} sentences captured, expected {len(sentences)}\n")

Doc 9:
HeteroData(
  sentence={ x=[27, 128] },
  paragraph={ x=[6, 128] },
  section={ x=[2, 128] },
  document={ x=[1, 128] },
  (sentence, belongs_to, paragraph)={ edge_index=[2, 27] },
  (paragraph, belongs_to, section)={ edge_index=[2, 6] },
  (section, belongs_to, document)={ edge_index=[2, 2] }
)
Sanity check: 27 sentences captured, expected 27

Doc 3:
HeteroData(
  sentence={ x=[175, 128] },
  paragraph={ x=[35, 128] },
  section={ x=[9, 128] },
  document={ x=[1, 128] },
  (sentence, belongs_to, paragraph)={ edge_index=[2, 175] },
  (paragraph, belongs_to, section)={ edge_index=[2, 35] },
  (section, belongs_to, document)={ edge_index=[2, 9] }
)
Sanity check: 175 sentences captured, expected 175



In [ ]:
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded. Embedding dimension:", embedder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 384


/tmp/ipykernel_1881/1372287890.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model loaded. Embedding dimension:", embedder.get_sentence_embedding_dimension())


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def build_semantic_edges(sentence_list, embedder, threshold=0.6, max_edges_per_node=5):

    embeddings = embedder.encode(sentence_list)
    sim_matrix = cosine_similarity(embeddings)

    src, dst = [], []
    n = len(sentence_list)
    for i in range(n):
        sims = sim_matrix[i].copy()
        sims[i] = -1
        top_k_idx = np.argsort(sims)[::-1][:max_edges_per_node]
        for j in top_k_idx:
            if sims[j] >= threshold:
                src.append(i)
                dst.append(j)

    return torch.tensor([src, dst]), sim_matrix

# Test on Doc 0
sentences = split_sentences(dataset[0]["article"])
edge_index, sim_matrix = build_semantic_edges(sentences, embedder)

print(f"Number of semantic edges created: {edge_index.shape[1]}")
print(f"Similarity score range: min={sim_matrix.min():.3f}, max (excl. self)={np.sort(sim_matrix.flatten())[-2]:.3f}")
print(f"Similarity score distribution: mean={sim_matrix.mean():.3f}, median={np.median(sim_matrix):.3f}")

Number of semantic edges created: 447
Similarity score range: min=-0.108, max (excl. self)=1.000
Similarity score distribution: mean=0.280, median=0.262


In [ ]:
sentences_long = split_sentences(dataset[3]["article"])  # 175 sentences
edge_index_long, sim_matrix_long = build_semantic_edges(sentences_long, embedder)

print(f"Doc 3 (175 sentences): {edge_index_long.shape[1]} semantic edges created")
print(f"Similarity distribution: mean={sim_matrix_long.mean():.3f}, median={np.median(sim_matrix_long):.3f}")

Doc 3 (175 sentences): 632 semantic edges created
Similarity distribution: mean=0.342, median=0.321


In [ ]:
def build_full_graph(article_text, embedder, embedding_dim=384, threshold=0.6, max_edges_per_node=5):
    """
    Full pipeline: raw article text -> HeteroData graph with structural + semantic edges.
    Uses real sentence embeddings (not random) as node features this time.
    """
    sentences = split_sentences(article_text)
    hierarchy = build_hierarchy(sentences)
    graph, sentence_list = build_graph(hierarchy, embedding_dim=embedding_dim)

    real_embeddings = embedder.encode(sentence_list)
    graph['sentence'].x = torch.tensor(real_embeddings, dtype=torch.float)

    sem_edge_index, _ = build_semantic_edges(sentence_list, embedder, threshold, max_edges_per_node)
    graph['sentence', 'similar_to', 'sentence'].edge_index = sem_edge_index

    return graph

graph = build_full_graph(dataset[0]["article"], embedder)
print(graph)

HeteroData(
  sentence={ x=[153, 384] },
  paragraph={ x=[31, 384] },
  section={ x=[8, 384] },
  document={ x=[1, 384] },
  (sentence, belongs_to, paragraph)={ edge_index=[2, 153] },
  (paragraph, belongs_to, section)={ edge_index=[2, 31] },
  (section, belongs_to, document)={ edge_index=[2, 8] },
  (sentence, similar_to, sentence)={ edge_index=[2, 447] }
)


In [ ]:
for i in range(len(graph_dataset)):
    article = dataset_50[i]["article"]
    sentences = split_sentences(article)
    if len(sentences) == 0:
        print(f"Doc {i}: EMPTY — 0 sentences. Raw article length: {len(article)} chars")
    elif len(sentences) < 3:
        print(f"Doc {i}: very short — {len(sentences)} sentences: {sentences}")

NameError: name 'graph_dataset' is not defined

In [ ]:
def build_semantic_edges(sentence_list, embedder, threshold=0.6, max_edges_per_node=5):
    n = len(sentence_list)

    if n < 2:
        return torch.tensor([[], []], dtype=torch.long), None

    embeddings = embedder.encode(sentence_list)
    sim_matrix = cosine_similarity(embeddings)

    src, dst = [], []
    for i in range(n):
        sims = sim_matrix[i].copy()
        sims[i] = -1
        top_k_idx = np.argsort(sims)[::-1][:max_edges_per_node]
        for j in top_k_idx:
            if sims[j] >= threshold:
                src.append(i)
                dst.append(j)

    return torch.tensor([src, dst]), sim_matrix

In [ ]:
class GraphSummarizationDataset(TorchDataset):
    def __init__(self, hf_dataset, embedder, embedding_dim=384, threshold=0.6, max_edges_per_node=5):
        self.embedder = embedder
        self.embedding_dim = embedding_dim
        self.threshold = threshold
        self.max_edges_per_node = max_edges_per_node

        self.valid_indices = [
            i for i in range(len(hf_dataset))
            if len(hf_dataset[i]["article"].strip()) > 0
        ]
        self.hf_dataset = hf_dataset
        print(f"Filtered dataset: {len(self.valid_indices)}/{len(hf_dataset)} documents have non-empty articles")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        real_idx = self.valid_indices[idx]
        article = self.hf_dataset[real_idx]["article"]
        summary = self.hf_dataset[real_idx]["abstract"]
        graph = build_full_graph(
            article, self.embedder,
            embedding_dim=self.embedding_dim,
            threshold=self.threshold,
            max_edges_per_node=self.max_edges_per_node
        )
        return graph, summary

In [ ]:
graph_dataset = GraphSummarizationDataset(dataset_50, embedder)
print(f"Dataset size: {len(graph_dataset)}")

node_counts = []
edge_counts = []
for i in range(len(graph_dataset)):
    graph, summary = graph_dataset[i]
    node_counts.append(graph['sentence'].x.shape[0])
    edge_counts.append(graph['sentence', 'similar_to', 'sentence'].edge_index.shape[1])
    if i % 10 == 0:
        print(f"Processed doc {i}: {node_counts[-1]} sentences, {edge_counts[-1]} semantic edges")

print(f"\nAll {len(graph_dataset)} documents processed successfully.")
print(f"Sentence count range: {min(node_counts)}–{max(node_counts)}")
print(f"Semantic edge count range: {min(edge_counts)}–{max(edge_counts)}")

Week3


In [ ]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
model = model.to(device)

print(f"Model loaded on: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Model loaded on: cuda
Model parameters: 60,506,624


In [ ]:
def tokenize_batch(articles, summaries, tokenizer, max_input_len=512, max_target_len=150):
    inputs = ["summarize: " + a for a in articles]
    model_inputs = tokenizer(
        inputs,
        max_length=max_input_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    labels = tokenizer(
        summaries,
        max_length=max_target_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


test_articles = [dataset_50[0]["article"], dataset_50[1]["article"]]
test_summaries = [dataset_50[0]["abstract"], dataset_50[1]["abstract"]]

batch = tokenize_batch(test_articles, test_summaries, tokenizer)
print("Input IDs shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)

Input IDs shape: torch.Size([2, 512])
Labels shape: torch.Size([2, 150])


In [ ]:
train_data = load_dataset("ccdv/pubmed-summarization", split="train[:1000]")
train_data = train_data.filter(lambda x: len(x["article"].strip()) > 0)
print(f"Training set size after filtering: {len(train_data)}")

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Training set size after filtering: 983


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
import time

class SummarizationDataset(TorchDataset):
    def __init__(self, hf_dataset, tokenizer, max_input_len=512, max_target_len=150):
        self.hf_dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        article = self.hf_dataset[idx]["article"]
        summary = self.hf_dataset[idx]["abstract"]

        inputs = self.tokenizer(
            "summarize: " + article,
            max_length=self.max_input_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        labels = self.tokenizer(
            summary,
            max_length=self.max_target_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "labels": labels["input_ids"].squeeze(0)
        }

train_dataset = SummarizationDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print(f"Training batches per epoch: {len(train_loader)}")

Training batches per epoch: 246


In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()

num_epochs = 1
log_every = 20

start_time = time.time()

for epoch in range(num_epochs):
    total_loss = 0
    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if step % log_every == 0:
            elapsed = time.time() - start_time
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {loss.item():.4f} | Elapsed: {elapsed:.1f}s")

    avg_loss = total_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} complete. Average loss: {avg_loss:.4f}")

print(f"\nTotal training time: {time.time() - start_time:.1f}s")

Epoch 1 | Step 0/246 | Loss: 6.3999 | Elapsed: 0.9s
Epoch 1 | Step 20/246 | Loss: 3.8759 | Elapsed: 6.2s
Epoch 1 | Step 40/246 | Loss: 4.3471 | Elapsed: 11.8s
Epoch 1 | Step 60/246 | Loss: 2.9819 | Elapsed: 17.4s
Epoch 1 | Step 80/246 | Loss: 2.2924 | Elapsed: 22.6s
Epoch 1 | Step 100/246 | Loss: 2.9061 | Elapsed: 27.9s
Epoch 1 | Step 120/246 | Loss: 2.9298 | Elapsed: 33.3s
Epoch 1 | Step 140/246 | Loss: 2.5051 | Elapsed: 41.6s
Epoch 1 | Step 160/246 | Loss: 3.0538 | Elapsed: 47.3s
Epoch 1 | Step 180/246 | Loss: 2.8525 | Elapsed: 52.7s
Epoch 1 | Step 200/246 | Loss: 2.6985 | Elapsed: 58.3s
Epoch 1 | Step 220/246 | Loss: 3.2884 | Elapsed: 63.9s
Epoch 1 | Step 240/246 | Loss: 3.2220 | Elapsed: 69.9s

Epoch 1 complete. Average loss: 3.3019

Total training time: 71.3s


In [ ]:
model.eval()

def generate_summary(article, model, tokenizer, max_input_len=512, max_output_len=150):
    inputs = tokenizer(
        "summarize: " + article,
        max_length=max_input_len,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=max_output_len,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

test_examples = train_data.select(range(5))

predictions = []
references = []

for i in range(5):
    article = test_examples[i]["article"]
    reference = test_examples[i]["abstract"]
    prediction = generate_summary(article, model, tokenizer)

    predictions.append(prediction)
    references.append(reference)

    print(f"--- Example {i} ---")
    print("PREDICTION:", prediction[:300])
    print("REFERENCE: ", reference[:300])
    print()

--- Example 0 ---
PREDICTION: anthropometric study of 752 high school girls in sistan and baluchestan showed prevalence of 16.2%, 8.6% and 1.5%, for underweight, overweight and obesity, respectively. in vietnam, school base program ( nffp ) is implemented in elementary schools of deprived areas to cover all poor students.
REFERENCE:  background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutritional intervention has been done between 2008 and

--- Example 1 ---
PREDICTION: anemia is defined as an inadequate circulating level of hemoglobin ( hb ) ( hb  12 g / dl ) and may arise as a result of the underlying disease, bleeding, poor nutrition, chemotherapy, or radiation therapy. preliminary studies suggest that survival and loco - regional control after radiation therapy
REFERENCE:  backgroundanemia in

In [ ]:
!pip install evaluate rouge_score bert_score -q


In [ ]:
import evaluate

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

rouge_results = rouge.compute(predictions=predictions, references=references)
print("ROUGE scores:")
for k, v in rouge_results.items():
    print(f"  {k}: {v:.4f}")

bertscore_results = bertscore.compute(predictions=predictions, references=references, lang="en")
print("\nBERTScore:")
print(f"  Precision: {sum(bertscore_results['precision'])/len(bertscore_results['precision']):.4f}")
print(f"  Recall:    {sum(bertscore_results['recall'])/len(bertscore_results['recall']):.4f}")
print(f"  F1:        {sum(bertscore_results['f1'])/len(bertscore_results['f1']):.4f}")

ROUGE scores:
  rouge1: 0.2201
  rouge2: 0.0324
  rougeL: 0.1076
  rougeLsum: 0.1823


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERTScore:
  Precision: 0.8529
  Recall:    0.8064
  F1:        0.8289


In [ ]:
import os
import pickle
import time

CACHE_DIR = "/content/embedding_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def get_cached_embeddings(doc_id, sentences, embedder, cache_dir=CACHE_DIR):
    """
    Returns sentence embeddings for a document, using a cache file if it exists,
    otherwise computes and saves them.
    """
    cache_path = os.path.join(cache_dir, f"doc_{doc_id}.pkl")

    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            embeddings = pickle.load(f)
    else:
        embeddings = embedder.encode(sentences)
        with open(cache_path, "wb") as f:
            pickle.dump(embeddings, f)

    return embeddings



for doc_id in range(5):
    sentences = split_sentences(dataset_50[doc_id]["article"])

    start = time.time()
    emb1 = get_cached_embeddings(doc_id, sentences, embedder)
    first_call_time = time.time() - start

    start = time.time()
    emb2 = get_cached_embeddings(doc_id, sentences, embedder)  # should hit cache now
    second_call_time = time.time() - start

    print(f"Doc {doc_id}: first call {first_call_time:.3f}s, second call (cached) {second_call_time:.3f}s, shapes match: {emb1.shape == emb2.shape}")

Doc 0: first call 0.146s, second call (cached) 0.000s, shapes match: True
Doc 1: first call 0.096s, second call (cached) 0.000s, shapes match: True
Doc 2: first call 0.047s, second call (cached) 0.000s, shapes match: True
Doc 3: first call 0.151s, second call (cached) 0.000s, shapes match: True
Doc 4: first call 0.054s, second call (cached) 0.000s, shapes match: True


In [ ]:
Week 4

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, HeteroConv

class HierarchicalGATLayer(nn.Module):

    def __init__(self, in_channels, out_channels, heads=4):
        super().__init__()
        self.conv = HeteroConv({
            ('sentence', 'belongs_to', 'paragraph'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('paragraph', 'belongs_to', 'section'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('section', 'belongs_to', 'document'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('sentence', 'similar_to', 'sentence'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
        }, aggr='sum')

    def forward(self, x_dict, edge_index_dict):
        out_dict = self.conv(x_dict, edge_index_dict)
        # HeteroConv only returns node types that received messages this layer;
        # carry forward untouched node types unchanged (important for sparse graphs)
        for node_type in x_dict:
            if node_type not in out_dict:
                out_dict[node_type] = x_dict[node_type]
        return {k: F.elu(v) for k, v in out_dict.items()}

In [ ]:
# Built a real graph to test on
graph = build_full_graph(dataset_50[0]["article"], embedder)
print("Input graph:")
print(graph)

layer = HierarchicalGATLayer(in_channels=-1, out_channels=64, heads=4).to(device)
graph = graph.to(device)

x_dict = graph.x_dict
edge_index_dict = graph.edge_index_dict

out_dict = layer(x_dict, edge_index_dict)

print("\nOutput shapes after 1 GAT layer:")
for node_type, feat in out_dict.items():
    print(f"  {node_type}: {feat.shape}")

Input graph:
HeteroData(
  sentence={ x=[153, 384] },
  paragraph={ x=[31, 384] },
  section={ x=[8, 384] },
  document={ x=[1, 384] },
  (sentence, belongs_to, paragraph)={ edge_index=[2, 153] },
  (paragraph, belongs_to, section)={ edge_index=[2, 31] },
  (section, belongs_to, document)={ edge_index=[2, 8] },
  (sentence, similar_to, sentence)={ edge_index=[2, 447] }
)

Output shapes after 1 GAT layer:
  paragraph: torch.Size([31, 256])
  section: torch.Size([8, 256])
  document: torch.Size([1, 256])
  sentence: torch.Size([153, 256])


In [ ]:
class HierarchicalGAT(nn.Module):
    """
    Full 2-layer Hierarchical Graph Attention Network.
    Layer 1: raw embeddings -> hidden representations (upward flow begins)
    Layer 2: hidden -> refined representations (deeper propagation, document context can flow back down)
    """
    def __init__(self, hidden_channels=64, heads=4):
        super().__init__()
        self.layer1 = HierarchicalGATLayer(in_channels=-1, out_channels=hidden_channels, heads=heads)
        self.layer2 = HierarchicalGATLayer(in_channels=-1, out_channels=hidden_channels, heads=heads)

    def forward(self, x_dict, edge_index_dict):
        x_dict = self.layer1(x_dict, edge_index_dict)
        x_dict = self.layer2(x_dict, edge_index_dict)
        return x_dict

# Test on the same graph
model_gat = HierarchicalGAT(hidden_channels=64, heads=4).to(device)

out_dict = model_gat(graph.x_dict, graph.edge_index_dict)

print("Output shapes after 2 GAT layers:")
for node_type, feat in out_dict.items():
    print(f"  {node_type}: {feat.shape}")

Output shapes after 2 GAT layers:
  paragraph: torch.Size([31, 256])
  section: torch.Size([8, 256])
  document: torch.Size([1, 256])
  sentence: torch.Size([153, 256])


In [ ]:
class AttentionPooling(nn.Module):

    def __init__(self, in_channels):
        super().__init__()
        self.attn = nn.Linear(in_channels, 1)

    def forward(self, x):
        # x: [num_nodes, in_channels]
        scores = self.attn(x)                      # [num_nodes, 1]
        weights = F.softmax(scores, dim=0)          # normalize across nodes
        pooled = (weights * x).sum(dim=0)            # [in_channels]
        return pooled, weights

# pool over sentence nodes as an alternative document representation
pooling = AttentionPooling(in_channels=256).to(device)

sentence_reprs = out_dict['sentence']  # [153, 256]
pooled_doc_repr, attn_weights = pooling(sentence_reprs)

print("Pooled document representation shape:", pooled_doc_repr.shape)
print("Attention weights shape:", attn_weights.shape)
print("Attention weights sum to 1:", attn_weights.sum().item())
print("\nTop 5 most attended sentences (by weight):")
top5_idx = attn_weights.squeeze().topk(5).indices.tolist()
sentences = split_sentences(dataset_50[0]["article"])
for idx in top5_idx:
    print(f"  [{attn_weights[idx].item():.4f}] {sentences[idx][:100]}")

Pooled document representation shape: torch.Size([256])
Attention weights shape: torch.Size([153, 1])
Attention weights sum to 1: 1.0

Top 5 most attended sentences (by weight):
  [0.0070] educate fifteen talk show programs in tv and radio , 12 published papers in the local newspaper , ha
  [0.0070] educate fifteen talk show programs in tv and radio , 12 published papers in the local newspaper , ha
  [0.0070] for educate fifteen talk show programs in tv and radio , 12 published papers in the local newspaper 
  [0.0068] advocacy group formation : this step was started with stakeholder analysis and identifying the stake
  [0.0068] advocacy group formation : this step was started with stakeholder analysis and identifying the stake


In [ ]:
class HierarchicalGATWithPooling(nn.Module):

    def __init__(self, hidden_channels=64, heads=4):
        super().__init__()
        self.gat = HierarchicalGAT(hidden_channels=hidden_channels, heads=heads)
        self.pooling = AttentionPooling(in_channels=hidden_channels * heads)

    def forward(self, x_dict, edge_index_dict):
        out_dict = self.gat(x_dict, edge_index_dict)
        doc_repr, attn_weights = self.pooling(out_dict['sentence'])
        return doc_repr, attn_weights


model_full = HierarchicalGATWithPooling(hidden_channels=64, heads=4).to(device)

doc_repr, attn_weights = model_full(graph.x_dict, graph.edge_index_dict)
print("Document representation shape:", doc_repr.shape)

fake_loss = doc_repr.sum()
fake_loss.backward()

has_grad = False
zero_grad_params = 0
total_params = 0
for name, param in model_full.named_parameters():
    total_params += 1
    if param.grad is not None:
        has_grad = True
        if param.grad.abs().sum().item() == 0:
            zero_grad_params += 1

print(f"\nGradients computed successfully: {has_grad}")
print(f"Parameters with zero gradient: {zero_grad_params}/{total_params}")

Document representation shape: torch.Size([256])

Gradients computed successfully: True
Parameters with zero gradient: 0/42


Week 5


In [ ]:
class GraphToDecoderBridge(nn.Module):
    """
    Projects the GAT's document representation into a pseudo-encoder-hidden-state
    sequence that T5's decoder can cross-attend to.
    """
    def __init__(self, gat_dim=256, decoder_dim=512):
        super().__init__()
        self.projection = nn.Linear(gat_dim, decoder_dim)

    def forward(self, doc_repr):
        # doc_repr: [gat_dim] -> [1, 1, decoder_dim]  (batch=1, seq_len=1, hidden=512)
        projected = self.projection(doc_repr)
        pseudo_encoder_output = projected.unsqueeze(0).unsqueeze(0)
        return pseudo_encoder_output

# Test the bridge in isolation
bridge = GraphToDecoderBridge(gat_dim=256, decoder_dim=512).to(device)

doc_repr, _ = model_full(graph.x_dict, graph.edge_index_dict)
pseudo_output = bridge(doc_repr)

print("Doc repr shape:", doc_repr.shape)
print("Pseudo encoder output shape:", pseudo_output.shape)

Doc repr shape: torch.Size([256])
Pseudo encoder output shape: torch.Size([1, 1, 512])


In [ ]:
from transformers.modeling_outputs import BaseModelOutput


encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)

encoder_attention_mask = torch.ones(1, 1, dtype=torch.long).to(device)

target_summary = dataset_50[0]["abstract"]
target_ids = tokenizer(
    target_summary,
    max_length=150,
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)["input_ids"].to(device)

outputs = model(
    encoder_outputs=encoder_outputs,
    attention_mask=encoder_attention_mask,
    labels=target_ids
)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

Loss: 4.8777756690979
Logits shape: torch.Size([1, 150, 32128])


In [ ]:
class HierarchicalGATSummarizer(nn.Module):
    """
    Full end-to-end model: HeteroData graph -> GAT encoder -> attention pooling
    -> bridge projection -> T5 decoder -> summary logits.
    """
    def __init__(self, t5_model, gat_hidden=64, gat_heads=4, decoder_dim=512):
        super().__init__()
        self.gat_encoder = HierarchicalGATWithPooling(hidden_channels=gat_hidden, heads=gat_heads)
        self.bridge = GraphToDecoderBridge(gat_dim=gat_hidden * gat_heads, decoder_dim=decoder_dim)
        self.t5 = t5_model  # reuse the same T5-small instance (decoder + LM head)

    def forward(self, x_dict, edge_index_dict, labels=None):
        doc_repr, attn_weights = self.gat_encoder(x_dict, edge_index_dict)
        pseudo_output = self.bridge(doc_repr)  # [1, 1, decoder_dim]

        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, 1, dtype=torch.long, device=pseudo_output.device)

        outputs = self.t5(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            labels=labels
        )
        return outputs, attn_weights

# Quick sanity test before building the full loop
# NOTE: reload a fresh T5 to avoid mixing with the flat-baseline's fine-tuned weights
fresh_t5 = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)

full_model = HierarchicalGATSummarizer(fresh_t5).to(device)

outputs, attn_weights = full_model(graph.x_dict, graph.edge_index_dict, labels=target_ids)
print("Loss:", outputs.loss.item())

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loss: 8.180273056030273


In [ ]:
optimizer = AdamW(full_model.parameters(), lr=5e-5)
full_model.train()

# Build training graphs from your 983-doc filtered training set (reuse from Week 3)
# For Week 5's first pass, train on a smaller subset (e.g. 100 docs) to keep iteration fast
train_subset_size = 100
num_epochs = 1
log_every = 10

checkpoint_dir = "/content/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

start_time = time.time()
losses = []

for epoch in range(num_epochs):
    total_loss = 0
    valid_steps = 0

    for i in range(train_subset_size):
        article = train_data[i]["article"]
        summary = train_data[i]["abstract"]

        # Build graph for this document
        doc_graph = build_full_graph(article, embedder)
        doc_graph = doc_graph.to(device)

        # Tokenize target summary
        target = tokenizer(
            summary, max_length=150, truncation=True, padding="max_length", return_tensors="pt"
        )["input_ids"].to(device)

        # Forward + backward
        outputs, _ = full_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        valid_steps += 1
        losses.append(loss.item())

        if i % log_every == 0:
            elapsed = time.time() - start_time
            print(f"Step {i}/{train_subset_size} | Loss: {loss.item():.4f} | Elapsed: {elapsed:.1f}s")

    avg_loss = total_loss / valid_steps
    print(f"\nEpoch {epoch+1} complete. Average loss: {avg_loss:.4f}")

# Save checkpoint
torch.save(full_model.state_dict(), os.path.join(checkpoint_dir, "hgat_summarizer_epoch1.pt"))
print(f"\nCheckpoint saved. Total training time: {time.time() - start_time:.1f}s")

Step 0/100 | Loss: 8.1599 | Elapsed: 0.4s
Step 10/100 | Loss: 5.9284 | Elapsed: 2.8s
Step 20/100 | Loss: 5.6746 | Elapsed: 5.9s
Step 30/100 | Loss: 4.3952 | Elapsed: 9.1s
Step 40/100 | Loss: 4.7605 | Elapsed: 11.7s
Step 50/100 | Loss: 7.5847 | Elapsed: 16.4s
Step 60/100 | Loss: 4.7127 | Elapsed: 19.1s
Step 70/100 | Loss: 4.8205 | Elapsed: 21.3s
Step 80/100 | Loss: 5.3120 | Elapsed: 24.0s
Step 90/100 | Loss: 4.6128 | Elapsed: 26.7s

Epoch 1 complete. Average loss: 5.4202

Checkpoint saved. Total training time: 34.7s


In [ ]:
full_model.eval()

def generate_summary_gat(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)

    with torch.no_grad():
        doc_repr, attn_weights = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        pseudo_output = model.bridge(doc_repr)
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, 1, dtype=torch.long, device=device)

        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Test on 5 examples
for i in range(5):
    article = train_data[i]["article"]
    reference = train_data[i]["abstract"]
    prediction = generate_summary_gat(article, full_model, embedder, tokenizer)

    print(f"--- Example {i} ---")
    print("PREDICTION:", prediction[:300])
    print("REFERENCE: ", reference[:300])
    print()

--- Example 0 ---
PREDICTION:                                                                
REFERENCE:  background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutritional intervention has been done between 2008 and

--- Example 1 ---
PREDICTION:                                                               
REFERENCE:  backgroundanemia in patients with cancer who are undergoing active therapy is commonly encountered and may worsen quality of life in these patients . the effect of blood transfusion is often temporary and may be associated with serious adverse events . 
 erythropoiesis - stimulating agents are not e

--- Example 2 ---
PREDICTION:                                                                       
REFERENCE:  tardive dystonia ( td ) is a serious side effect of antipsychotic medica

In [ ]:
article = train_data[0]["article"]
doc_graph = build_full_graph(article, embedder).to(device)

full_model.eval()
with torch.no_grad():
    doc_repr, attn_weights = full_model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
    pseudo_output = full_model.bridge(doc_repr)
    encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
    encoder_attention_mask = torch.ones(1, 1, dtype=torch.long, device=device)

    summary_ids = full_model.t5.generate(
        encoder_outputs=encoder_outputs,
        attention_mask=encoder_attention_mask,
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

print("Raw token IDs:", summary_ids)
print("Token ID meanings:")
for tid in summary_ids[0][:10]:
    print(f"  {tid.item()}: '{tokenizer.decode([tid])}'")
print("\nPad token id:", tokenizer.pad_token_id)
print("EOS token id:", tokenizer.eos_token_id)

Raw token IDs: tensor([[0, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 1]], device='cuda:0')
Token ID meanings:
  0: '<pad>'
  3: ''
  2: '<unk>'
  3: ''
  2: '<unk>'
  3: ''
  2: '<unk>'
  3: ''
  2: '<unk>'
  3: ''

Pad token id: 0
EOS token id: 1


In [ ]:
def tokenize_targets(summaries, tokenizer, max_target_len=150):
    labels = tokenizer(
        summaries,
        max_length=max_target_len,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    )["input_ids"]

    # Replace pad token id with -100 so loss ignores padding positions
    labels[labels == tokenizer.pad_token_id] = -100
    return labels

In [ ]:
fresh_t5 = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
full_model = HierarchicalGATSummarizer(fresh_t5).to(device)
optimizer = AdamW(full_model.parameters(), lr=5e-5)
full_model.train()

train_subset_size = 100
num_epochs = 1
log_every = 10
losses = []
start_time = time.time()

for epoch in range(num_epochs):
    total_loss = 0
    for i in range(train_subset_size):
        article = train_data[i]["article"]
        summary = train_data[i]["abstract"]

        doc_graph = build_full_graph(article, embedder).to(device)
        target = tokenize_targets([summary], tokenizer).to(device)

        outputs, _ = full_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        losses.append(loss.item())

        if i % log_every == 0:
            print(f"Step {i}/{train_subset_size} | Loss: {loss.item():.4f} | Elapsed: {time.time()-start_time:.1f}s")

avg_loss = total_loss / train_subset_size
print(f"\nEpoch complete. Average loss: {avg_loss:.4f}")
torch.save(full_model.state_dict(), os.path.join(checkpoint_dir, "hgat_summarizer_epoch1_fixed.pt"))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Step 0/100 | Loss: 8.3238 | Elapsed: 0.4s
Step 10/100 | Loss: 6.0493 | Elapsed: 2.7s
Step 20/100 | Loss: 5.8507 | Elapsed: 5.4s
Step 30/100 | Loss: 4.3398 | Elapsed: 8.3s
Step 40/100 | Loss: 4.6936 | Elapsed: 11.4s
Step 50/100 | Loss: 4.7861 | Elapsed: 14.2s
Step 60/100 | Loss: 4.4479 | Elapsed: 17.2s
Step 70/100 | Loss: 4.8245 | Elapsed: 19.0s
Step 80/100 | Loss: 5.1463 | Elapsed: 22.0s
Step 90/100 | Loss: 4.5050 | Elapsed: 25.1s

Epoch complete. Average loss: 5.2696


In [ ]:
full_model.eval()

def generate_summary_gat(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)

    with torch.no_grad():
        doc_repr, attn_weights = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        pseudo_output = model.bridge(doc_repr)
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, 1, dtype=torch.long, device=device)

        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

for i in range(5):
    article = train_data[i]["article"]
    reference = train_data[i]["abstract"]
    prediction = generate_summary_gat(article, full_model, embedder, tokenizer)

    print(f"--- Example {i} ---")
    print("PREDICTION:", prediction[:300])
    print("REFERENCE: ", reference[:300])
    print()

--- Example 0 ---
PREDICTION:               .
REFERENCE:  background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutritional intervention has been done between 2008 and

--- Example 1 ---
PREDICTION: ......
REFERENCE:  backgroundanemia in patients with cancer who are undergoing active therapy is commonly encountered and may worsen quality of life in these patients . the effect of blood transfusion is often temporary and may be associated with serious adverse events . 
 erythropoiesis - stimulating agents are not e

--- Example 2 ---
PREDICTION: .....
REFERENCE:  tardive dystonia ( td ) is a serious side effect of antipsychotic medications , more with typical antipsychotics , that is potentially irreversible in affected patients . 
 studies show that newer atypical antipsychotics have a lower risk of

In [ ]:
class GraphToDecoderBridge(nn.Module):

    def __init__(self, gat_dim=256, decoder_dim=512):
        super().__init__()
        self.projection = nn.Linear(gat_dim, decoder_dim)

    def forward(self, section_reprs):
        # section_reprs: [num_sections, gat_dim] -> [1, num_sections, decoder_dim]
        projected = self.projection(section_reprs)       # [num_sections, decoder_dim]
        pseudo_encoder_output = projected.unsqueeze(0)     # [1, num_sections, decoder_dim]
        return pseudo_encoder_output


class HierarchicalGATSummarizer(nn.Module):
    def __init__(self, t5_model, gat_hidden=64, gat_heads=4, decoder_dim=512):
        super().__init__()
        self.gat_encoder = HierarchicalGAT(hidden_channels=gat_hidden, heads=gat_heads)
        self.bridge = GraphToDecoderBridge(gat_dim=gat_hidden * gat_heads, decoder_dim=decoder_dim)
        self.t5 = t5_model

    def forward(self, x_dict, edge_index_dict, labels=None):
        out_dict = self.gat_encoder(x_dict, edge_index_dict)
        section_reprs = out_dict['section']  # [num_sections, gat_dim]

        pseudo_output = self.bridge(section_reprs)  # [1, num_sections, decoder_dim]
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=pseudo_output.device)

        outputs = self.t5(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            labels=labels
        )
        return outputs

In [ ]:
fresh_t5 = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
full_model = HierarchicalGATSummarizer(fresh_t5).to(device)
optimizer = AdamW(full_model.parameters(), lr=5e-5)
full_model.train()

train_subset_size = 100
log_every = 10
losses = []
start_time = time.time()

total_loss = 0
for i in range(train_subset_size):
    article = train_data[i]["article"]
    summary = train_data[i]["abstract"]

    doc_graph = build_full_graph(article, embedder).to(device)
    target = tokenize_targets([summary], tokenizer).to(device)

    outputs = full_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    losses.append(loss.item())

    if i % log_every == 0:
        print(f"Step {i}/{train_subset_size} | Loss: {loss.item():.4f} | Elapsed: {time.time()-start_time:.1f}s")

print(f"\nEpoch complete. Average loss: {total_loss/train_subset_size:.4f}")
torch.save(full_model.state_dict(), os.path.join(checkpoint_dir, "hgat_summarizer_epoch1_v2.pt"))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Step 0/100 | Loss: 7.0242 | Elapsed: 0.9s
Step 10/100 | Loss: 5.4924 | Elapsed: 3.5s
Step 20/100 | Loss: 5.5494 | Elapsed: 6.5s
Step 30/100 | Loss: 4.3307 | Elapsed: 9.5s
Step 40/100 | Loss: 4.5510 | Elapsed: 11.8s
Step 50/100 | Loss: 4.5661 | Elapsed: 14.3s
Step 60/100 | Loss: 4.5602 | Elapsed: 17.1s
Step 70/100 | Loss: 4.6390 | Elapsed: 19.4s
Step 80/100 | Loss: 5.3157 | Elapsed: 22.1s
Step 90/100 | Loss: 4.5224 | Elapsed: 24.9s

Epoch complete. Average loss: 5.1344


In [ ]:
full_model.eval()

def generate_summary_gat_v2(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)

    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]

        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)

        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=4,
            early_stopping=True
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

for i in range(5):
    article = train_data[i]["article"]
    reference = train_data[i]["abstract"]
    prediction = generate_summary_gat_v2(article, full_model, embedder, tokenizer)

    print(f"--- Example {i} ---")
    print("PREDICTION:", prediction[:300])
    print("REFERENCE: ", reference[:300])
    print()

--- Example 0 ---
PREDICTION:                                                                           
REFERENCE:  background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutritional intervention has been done between 2008 and

--- Example 1 ---
PREDICTION:                                                                           
REFERENCE:  backgroundanemia in patients with cancer who are undergoing active therapy is commonly encountered and may worsen quality of life in these patients . the effect of blood transfusion is often temporary and may be associated with serious adverse events . 
 erythropoiesis - stimulating agents are not e

--- Example 2 ---
PREDICTION:                                                                           
REFERENCE:  tardive dystonia ( td ) is a serious side eff

In [ ]:
article = train_data[0]["article"]
doc_graph = build_full_graph(article, embedder).to(device)

full_model.eval()
with torch.no_grad():
    out_dict = full_model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
    section_reprs = out_dict['section']
    pseudo_output = full_model.bridge(section_reprs)
    num_sections = section_reprs.shape[0]

    encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
    encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)

    summary_ids = full_model.t5.generate(
        encoder_outputs=encoder_outputs,
        attention_mask=encoder_attention_mask,
        max_length=150,
        num_beams=4,
        early_stopping=True
    )

print("Raw token IDs:", summary_ids)
print("Token meanings:")
for tid in summary_ids[0][:10]:
    print(f"  {tid.item()}: '{tokenizer.decode([tid])}'")

# Also check the scale of the pseudo encoder output — this may be the real culprit
print("\nPseudo encoder output stats:")
print("  mean:", pseudo_output.mean().item())
print("  std:", pseudo_output.std().item())
print("  min:", pseudo_output.min().item())
print("  max:", pseudo_output.max().item())

Raw token IDs: tensor([[0, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3, 2, 3,
         2, 3, 2, 3, 2, 3]], device='cuda:0')
Token meanings:
  0: '<pad>'
  3: ''
  2: '<unk>'
  3: ''
  2: '<unk>'
  3: ''
  2: '<unk>'
  3: ''
  2: '<unk>'
  3: ''

Pseudo encoder output stats:
  mean: -0.002023933455348015
  std: 0.04138212278485298
  min: -0.12285818159580231
  max: 0.11306273937225342


In [ ]:
class GraphToDecoderBridge(nn.Module):
    def __init__(self, gat_dim=256, decoder_dim=512):
        super().__init__()
        self.projection = nn.Linear(gat_dim, decoder_dim)
        self.norm = nn.LayerNorm(decoder_dim)

    def forward(self, section_reprs):
        projected = self.projection(section_reprs)
        normalized = self.norm(projected)
        pseudo_encoder_output = normalized.unsqueeze(0)
        return pseudo_encoder_output

In [ ]:
fresh_t5 = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
full_model = HierarchicalGATSummarizer(fresh_t5).to(device)
optimizer = AdamW(full_model.parameters(), lr=5e-5)
full_model.train()

train_subset_size = 100
log_every = 10
losses = []
start_time = time.time()

total_loss = 0
for i in range(train_subset_size):
    article = train_data[i]["article"]
    summary = train_data[i]["abstract"]

    doc_graph = build_full_graph(article, embedder).to(device)
    target = tokenize_targets([summary], tokenizer).to(device)

    outputs = full_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
    losses.append(loss.item())

    if i % log_every == 0:
        print(f"Step {i}/{train_subset_size} | Loss: {loss.item():.4f} | Elapsed: {time.time()-start_time:.1f}s")

print(f"\nEpoch complete. Average loss: {total_loss/train_subset_size:.4f}")
torch.save(full_model.state_dict(), os.path.join(checkpoint_dir, "hgat_summarizer_epoch1_v3.pt"))

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Step 0/100 | Loss: 11.7349 | Elapsed: 0.4s
Step 10/100 | Loss: 9.7870 | Elapsed: 3.6s
Step 20/100 | Loss: 9.3862 | Elapsed: 6.3s
Step 30/100 | Loss: 9.0571 | Elapsed: 9.0s
Step 40/100 | Loss: 8.0640 | Elapsed: 11.5s
Step 50/100 | Loss: 7.8118 | Elapsed: 14.4s
Step 60/100 | Loss: 7.0228 | Elapsed: 17.6s
Step 70/100 | Loss: 7.3061 | Elapsed: 19.5s
Step 80/100 | Loss: 7.9996 | Elapsed: 22.0s
Step 90/100 | Loss: 7.4173 | Elapsed: 24.8s

Epoch complete. Average loss: 8.5028


In [ ]:
# Check pseudo encoder output scale
article = train_data[0]["article"]
doc_graph = build_full_graph(article, embedder).to(device)

full_model.eval()
with torch.no_grad():
    out_dict = full_model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
    section_reprs = out_dict['section']
    pseudo_output = full_model.bridge(section_reprs)

print("Pseudo encoder output stats:")
print("  mean:", pseudo_output.mean().item())
print("  std:", pseudo_output.std().item())

# Check generation on 5 examples
def generate_summary_gat_v3(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print()
for i in range(5):
    article = train_data[i]["article"]
    reference = train_data[i]["abstract"]
    prediction = generate_summary_gat_v3(article, full_model, embedder, tokenizer)
    print(f"--- Example {i} ---")
    print("PREDICTION:", prediction[:200])
    print("REFERENCE: ", reference[:150])
    print()

Pseudo encoder output stats:
  mean: -8.03852453827858e-05
  std: 0.9976293444633484

--- Example 0 ---
PREDICTION:                                                                                                                                                     
REFERENCE:  background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition st

--- Example 1 ---
PREDICTION:                                                                                                                                                     
REFERENCE:  backgroundanemia in patients with cancer who are undergoing active therapy is commonly encountered and may worsen quality of life in these patients . 

--- Example 2 ---
PREDICTION:                                                                                                                                                     
REFERENCE:  tardive dystonia ( td ) is a serious 

In [ ]:
full_model.train()
extra_epochs = 5

for epoch in range(extra_epochs):
    total_loss = 0
    for i in range(train_subset_size):
        article = train_data[i]["article"]
        summary = train_data[i]["abstract"]

        doc_graph = build_full_graph(article, embedder).to(device)
        target = tokenize_targets([summary], tokenizer).to(device)

        outputs = full_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+2} | Avg loss: {total_loss/train_subset_size:.4f}")

Epoch 2 | Avg loss: 6.9642
Epoch 3 | Avg loss: 6.5993
Epoch 4 | Avg loss: 6.4333
Epoch 5 | Avg loss: 6.3196
Epoch 6 | Avg loss: 6.2103


In [ ]:
def generate_greedy(article, model, embedder, tokenizer, max_output_len=100):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=1,
            do_sample=False
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

full_model.eval()
for i in range(3):
    pred = generate_greedy(train_data[i]["article"], full_model, embedder, tokenizer)
    print(f"Doc {i}: '{pred[:150]}'")

In [ ]:
def generate_greedy(article, model, embedder, tokenizer, max_output_len=100):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=1,
            do_sample=False
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

full_model.eval()
for i in range(5):
    pred = generate_greedy(train_data[i]["article"], full_model, embedder, tokenizer)
    ref = train_data[i]["abstract"]
    print(f"--- Doc {i} ---")
    print("PREDICTION:", pred[:150])
    print("REFERENCE: ", ref[:150])
    print()

--- Doc 0 ---
PREDICTION:                                                                                                   
REFERENCE:  background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition st

--- Doc 1 ---
PREDICTION: s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s s 
REFERENCE:  backgroundanemia in patients with cancer who are undergoing active therapy is commonly encountered and may worsen quality of life in these patients . 

--- Doc 2 ---
PREDICTION:  a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a
REFERENCE:  tardive dystonia ( td ) is a serious side effect of antipsychotic medications , more with typical antipsychotics , that is potentially irreversible in

--- Doc 3 ---
PREDICTION:  a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a a
REFERENC

In [ ]:
def generate_greedy_penalized(article, model, embedder, tokenizer, max_output_len=100):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=1,
            do_sample=False,
            repetition_penalty=2.5,
            no_repeat_ngram_size=2
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

full_model.eval()
for i in range(5):
    pred = generate_greedy_penalized(train_data[i]["article"], full_model, embedder, tokenizer)
    print(f"Doc {i}: '{pred[:150]}'")

Doc 0: '. a s,  - o d  and  of e r i m t )  the n  in  to : p c / y g l' for ly  ( ing  with  study  from re x h u th b based  were  an  on  was  is '
Doc 1: 's a. oss, - cs of l d ms and  i rs in es the ps to gs ( ) t y n / b fsis  is  with resas for : '
Doc 2: 'a, s.  oa the - i m t  and  of d e n c ra,, the the, to is l g  in re )  is  treatment / ly th y f p  (  patients  an :  with al'patient  patient er c'
Doc 3: 'a s i e,. n - d t o  of   and  in c r )  the p m re er  to l b cyto :  is  with y g en he ic  ( h in ing  for induced ; '
Doc 4: 'a sas  ia,. daa the oa of - ea and ra in ma to na an ca ( ta with ga is,ss, the thes of thea.s the ofs. the,,a was h rea for ) l pais fa as ly y ve '


In [ ]:
def build_semantic_edges(sentence_list, embedder, threshold=0.6, max_edges_per_node=5):
    n = len(sentence_list)

    if n < 2:
        return torch.tensor([[], []], dtype=torch.long), None

    embeddings = embedder.encode(sentence_list)
    sim_matrix = cosine_similarity(embeddings)

    src, dst = [], []
    for i in range(n):
        sims = sim_matrix[i].copy()
        sims[i] = -1
        top_k_idx = np.argsort(sims)[::-1][:max_edges_per_node]
        for j in top_k_idx:
            if sims[j] >= threshold:
                src.append(i)
                dst.append(j)

    # Fix: explicitly specify dtype=torch.long, since empty src/dst lists
    # would otherwise default to float32, breaking PyG's edge_index requirement
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    return edge_index, sim_matrix

In [ ]:
fresh_t5 = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
gat_model = HierarchicalGATSummarizer(fresh_t5).to(device)
optimizer = AdamW(gat_model.parameters(), lr=5e-5)
gat_model.train()

num_epochs = 3
train_size = len(train_data)
log_every = 100

start_time = time.time()
epoch_losses = []

for epoch in range(num_epochs):
    total_loss = 0
    for i in range(train_size):
        article = train_data[i]["article"]
        summary = train_data[i]["abstract"]

        doc_graph = build_full_graph(article, embedder).to(device)
        target = tokenize_targets([summary], tokenizer).to(device)

        outputs = gat_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if i % log_every == 0:
            elapsed = time.time() - start_time
            print(f"Epoch {epoch+1} | Step {i}/{train_size} | Loss: {loss.item():.4f} | Elapsed: {elapsed/60:.1f}min")

    avg_loss = total_loss / train_size
    epoch_losses.append(avg_loss)
    print(f"\n=== Epoch {epoch+1} complete. Avg loss: {avg_loss:.4f} ===\n")

torch.save(gat_model.state_dict(), os.path.join(checkpoint_dir, "hgat_summarizer_final.pt"))
print(f"Total training time: {(time.time()-start_time)/60:.1f} minutes")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1 | Step 0/983 | Loss: 12.9239 | Elapsed: 0.0min
Epoch 1 | Step 100/983 | Loss: 7.1045 | Elapsed: 0.4min
Epoch 1 | Step 200/983 | Loss: 7.2947 | Elapsed: 0.9min
Epoch 1 | Step 300/983 | Loss: 7.5659 | Elapsed: 1.3min
Epoch 1 | Step 400/983 | Loss: 6.6837 | Elapsed: 1.8min
Epoch 1 | Step 500/983 | Loss: 6.4846 | Elapsed: 2.2min
Epoch 1 | Step 600/983 | Loss: 5.8926 | Elapsed: 2.6min
Epoch 1 | Step 700/983 | Loss: 6.1211 | Elapsed: 3.1min
Epoch 1 | Step 800/983 | Loss: 6.2221 | Elapsed: 3.5min
Epoch 1 | Step 900/983 | Loss: 5.8549 | Elapsed: 4.0min

=== Epoch 1 complete. Avg loss: 6.6899 ===

Epoch 2 | Step 0/983 | Loss: 6.2805 | Elapsed: 4.3min
Epoch 2 | Step 100/983 | Loss: 5.5183 | Elapsed: 4.8min
Epoch 2 | Step 200/983 | Loss: 5.9991 | Elapsed: 5.3min
Epoch 2 | Step 300/983 | Loss: 6.7773 | Elapsed: 5.7min
Epoch 2 | Step 400/983 | Loss: 5.8618 | Elapsed: 6.2min
Epoch 2 | Step 500/983 | Loss: 5.6613 | Elapsed: 6.6min
Epoch 2 | Step 600/983 | Loss: 4.9719 | Elapsed: 7.0min
Epoch 

In [ ]:
gat_model.eval()

def generate_summary_final(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

for i in range(5):
    article = train_data[i]["article"]
    reference = train_data[i]["abstract"]
    prediction = generate_summary_final(article, gat_model, embedder, tokenizer)
    print(f"--- Example {i} ---")
    print("PREDICTION:", prediction[:250])
    print("REFERENCE: ", reference[:250])
    print()

--- Example 0 ---
PREDICTION: ,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,.,,,,,,,,,,,,,,,,, 
REFERENCE:  background : the present study was carried out to assess the effects of community nutrition intervention based on advocacy approach on malnutrition status among school - aged children in shiraz , iran.materials and methods : this case - control nutri

--- Example 1 ---
PREDICTION: ........................................................,................. 
REFERENCE:  backgroundanemia in patients with cancer who are undergoing active therapy is commonly encountered and may worsen quality of life in these patients . the effect of blood transfusion is often temporary and may be associated with serious adverse events

--- Example 2 ---
PREDICTION: ,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,, 
REFERENCE:  tardive dystonia ( td ) is a serious side effect of antipsychotic medications , more with typical antipsychotics , that is potentially irrever

In [ ]:
def generate_with_penalty(article, model, embedder, tokenizer, max_output_len=100):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=1,
            do_sample=False,
            repetition_penalty=2.5,
            no_repeat_ngram_size=2
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

gat_model.eval()
for i in range(5):
    pred = generate_with_penalty(train_data[i]["article"], gat_model, embedder, tokenizer)
    print(f"Doc {i}: '{pred[:150]}'")

Doc 0: ', the study of the literature is a study on the health and health of women in the society. we report that the research was conducted by the university'
Doc 1: ', a study of the clinical trials of patients with sclerosis ( ) and recurrent.'
Doc 2: ', a new study of the psychiopathic. it is based on the findings of sympathetic - etiology and regenerative pulmonary disease ( cnr ) in the literature'
Doc 3: ', and the role of the human body is to be a key role in the development of human cells. this study was characterized by the potential for regenerative'
Doc 4: 'acute pulmonary artery is characterized by septic fibromyal ventricular. the onset of cdothoracic thrombosis is the most common cause of acute ephromy'


In [ ]:
test_data = load_dataset("ccdv/pubmed-summarization", split="test[:50]")
test_data = test_data.filter(lambda x: len(x["article"].strip()) > 0)
print(f"Test set size after filtering: {len(test_data)}")

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

Test set size after filtering: 50


In [1]:
import evaluate

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def generate_baseline_summary(article, model, tokenizer, max_input_len=512, max_output_len=150):
    inputs = tokenizer(
        "summarize: " + article,
        max_length=max_input_len, truncation=True, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"], attention_mask=inputs["attention_mask"],
            max_length=max_output_len, num_beams=4, early_stopping=True
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def generate_gat_summary(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=encoder_attention_mask,
            max_length=max_output_len,
            num_beams=1, do_sample=False,
            repetition_penalty=2.5, no_repeat_ngram_size=2
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# Generate predictions from both models on the same 50 test documents
baseline_preds, gat_preds, references = [], [], []

model.eval()       # your Week 3 baseline
gat_model.eval()   # your Week 6 trained GAT model

print("Generating summaries from both models...")
for i in range(len(test_data)):
    article = test_data[i]["article"]
    reference = test_data[i]["abstract"]

    baseline_pred = generate_baseline_summary(article, model, tokenizer)
    gat_pred = generate_gat_summary(article, gat_model, embedder, tokenizer)

    baseline_preds.append(baseline_pred)
    gat_preds.append(gat_pred)
    references.append(reference)

    if i % 10 == 0:
        print(f"  Processed {i}/{len(test_data)}")

print("Done generating. Computing metrics...")

# ROUGE for both
baseline_rouge = rouge.compute(predictions=baseline_preds, references=references)
gat_rouge = rouge.compute(predictions=gat_preds, references=references)

# BERTScore for both
baseline_bert = bertscore.compute(predictions=baseline_preds, references=references, lang="en")
gat_bert = bertscore.compute(predictions=gat_preds, references=references, lang="en")

print("\n=== RESULTS TABLE ===")
print(f"{'Metric':<15} {'Baseline':<12} {'Hierarchical GAT':<18}")
print(f"{'ROUGE-1':<15} {baseline_rouge['rouge1']:<12.4f} {gat_rouge['rouge1']:<18.4f}")
print(f"{'ROUGE-2':<15} {baseline_rouge['rouge2']:<12.4f} {gat_rouge['rouge2']:<18.4f}")
print(f"{'ROUGE-L':<15} {baseline_rouge['rougeL']:<12.4f} {gat_rouge['rougeL']:<18.4f}")
print(f"{'BERTScore F1':<15} {sum(baseline_bert['f1'])/len(baseline_bert['f1']):<12.4f} {sum(gat_bert['f1'])/len(gat_bert['f1']):<18.4f}")

ModuleNotFoundError: No module named 'evaluate'

In [ ]:
!pip install --upgrade --force-reinstall bert-score -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 576.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 837.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import importlib
import evaluate

# Reload evaluate's bertscore module to pick up the upgraded library
bertscore = evaluate.load("bertscore")

# Retry just the BERTScore computation (ROUGE already succeeded, no need to redo it)
baseline_bert = bertscore.compute(predictions=baseline_preds, references=references, lang="en")
gat_bert = bertscore.compute(predictions=gat_preds, references=references, lang="en")

print("\n=== RESULTS TABLE ===")
print(f"{'Metric':<15} {'Baseline':<12} {'Hierarchical GAT':<18}")
print(f"{'ROUGE-1':<15} {baseline_rouge['rouge1']:<12.4f} {gat_rouge['rouge1']:<18.4f}")
print(f"{'ROUGE-2':<15} {baseline_rouge['rouge2']:<12.4f} {gat_rouge['rouge2']:<18.4f}")
print(f"{'ROUGE-L':<15} {baseline_rouge['rougeL']:<12.4f} {gat_rouge['rougeL']:<18.4f}")
print(f"{'BERTScore F1':<15} {sum(baseline_bert['f1'])/len(baseline_bert['f1']):<12.4f} {sum(gat_bert['f1'])/len(gat_bert['f1']):<18.4f}")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AttributeError: RobertaTokenizer has no attribute build_inputs_with_special_tokens

In [ ]:
baseline_bert = bertscore.compute(
    predictions=baseline_preds, references=references,
    lang="en", model_type="distilbert-base-uncased"
)
gat_bert = bertscore.compute(
    predictions=gat_preds, references=references,
    lang="en", model_type="distilbert-base-uncased"
)

print("\n=== RESULTS TABLE ===")
print(f"{'Metric':<15} {'Baseline':<12} {'Hierarchical GAT':<18}")
print(f"{'ROUGE-1':<15} {baseline_rouge['rouge1']:<12.4f} {gat_rouge['rouge1']:<18.4f}")
print(f"{'ROUGE-2':<15} {baseline_rouge['rouge2']:<12.4f} {gat_rouge['rouge2']:<18.4f}")
print(f"{'ROUGE-L':<15} {baseline_rouge['rougeL']:<12.4f} {gat_rouge['rougeL']:<18.4f}")
print(f"{'BERTScore F1':<15} {sum(baseline_bert['f1'])/len(baseline_bert['f1']):<12.4f} {sum(gat_bert['f1'])/len(gat_bert['f1']):<18.4f}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: BertTokenizer has no attribute build_inputs_with_special_tokens

In [ ]:
!pip install --upgrade --force-reinstall bert-score -q

In [ ]:
print("=== RESULTS TABLE (ROUGE only — BERTScore blocked by environment issue) ===")
print(f"{'Metric':<15} {'Baseline':<12} {'Hierarchical GAT':<18}")
print(f"{'ROUGE-1':<15} {baseline_rouge['rouge1']:<12.4f} {gat_rouge['rouge1']:<18.4f}")
print(f"{'ROUGE-2':<15} {baseline_rouge['rouge2']:<12.4f} {gat_rouge['rouge2']:<18.4f}")
print(f"{'ROUGE-L':<15} {baseline_rouge['rougeL']:<12.4f} {gat_rouge['rougeL']:<18.4f}")

=== RESULTS TABLE (ROUGE only — BERTScore blocked by environment issue) ===
Metric          Baseline     Hierarchical GAT  
ROUGE-1         0.2280       0.1830            
ROUGE-2         0.0901       0.0251            
ROUGE-L         0.1627       0.1103            


In [ ]:
import json
with open("/content/checkpoints/week6_predictions.json", "w") as f:
    json.dump({
        "baseline_preds": baseline_preds,
        "gat_preds": gat_preds,
        "references": references
    }, f)
print("Predictions saved.")

Predictions saved.


In [2]:
import os
print(os.path.exists("/content/checkpoints/hgat_summarizer_final.pt"))

False


In [2]:
!pip install torch-geometric transformers datasets evaluate rouge-score bert-score spacy sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.7 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
import time
import json
from datasets import load_dataset
from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv, HeteroConv
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch.optim import AdamW
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- Week 1/2 functions ---
def split_sentences(article_text):
    return [s.strip() for s in article_text.split("\n") if s.strip()]

def build_hierarchy(sentences, sentences_per_paragraph=5, paragraphs_per_section=4):
    paragraphs = [sentences[i:i+sentences_per_paragraph] for i in range(0, len(sentences), sentences_per_paragraph)]
    sections = [paragraphs[i:i+paragraphs_per_section] for i in range(0, len(paragraphs), paragraphs_per_section)]
    return sections

def build_graph(hierarchy, embedding_dim=128):
    data = HeteroData()
    sentence_list, sent_to_para, para_to_section = [], [], []
    para_counter = 0
    for section_idx, section in enumerate(hierarchy):
        for para in section:
            for sent in para:
                sentence_list.append(sent)
                sent_to_para.append(para_counter)
            para_to_section.append(section_idx)
            para_counter += 1
    num_sentences, num_paragraphs, num_sections = len(sentence_list), para_counter, len(hierarchy)
    data['sentence'].x = torch.randn(num_sentences, embedding_dim)
    data['paragraph'].x = torch.randn(num_paragraphs, embedding_dim)
    data['section'].x = torch.randn(num_sections, embedding_dim)
    data['document'].x = torch.randn(1, embedding_dim)
    data['sentence','belongs_to','paragraph'].edge_index = torch.tensor([list(range(num_sentences)), sent_to_para])
    data['paragraph','belongs_to','section'].edge_index = torch.tensor([list(range(num_paragraphs)), para_to_section])
    data['section','belongs_to','document'].edge_index = torch.tensor([list(range(num_sections)), [0]*num_sections])
    return data, sentence_list

def build_semantic_edges(sentence_list, embedder, threshold=0.6, max_edges_per_node=5):
    n = len(sentence_list)
    if n < 2:
        return torch.tensor([[], []], dtype=torch.long), None
    embeddings = embedder.encode(sentence_list)
    sim_matrix = cosine_similarity(embeddings)
    src, dst = [], []
    for i in range(n):
        sims = sim_matrix[i].copy()
        sims[i] = -1
        top_k_idx = np.argsort(sims)[::-1][:max_edges_per_node]
        for j in top_k_idx:
            if sims[j] >= threshold:
                src.append(i); dst.append(j)
    edge_index = torch.tensor([src, dst], dtype=torch.long)  # dtype fix
    return edge_index, sim_matrix

def build_full_graph(article_text, embedder, embedding_dim=384, threshold=0.6, max_edges_per_node=5):
    sentences = split_sentences(article_text)
    hierarchy = build_hierarchy(sentences)
    graph, sentence_list = build_graph(hierarchy, embedding_dim=embedding_dim)
    real_embeddings = embedder.encode(sentence_list)
    graph['sentence'].x = torch.tensor(real_embeddings, dtype=torch.float)
    sem_edge_index, _ = build_semantic_edges(sentence_list, embedder, threshold, max_edges_per_node)
    graph['sentence','similar_to','sentence'].edge_index = sem_edge_index
    return graph

def tokenize_targets(summaries, tokenizer, max_target_len=150):
    labels = tokenizer(summaries, max_length=max_target_len, truncation=True, padding="max_length", return_tensors="pt")["input_ids"]
    labels[labels == tokenizer.pad_token_id] = -100
    return labels

# --- Model classes ---
class HierarchicalGATLayer(nn.Module):
    def __init__(self, in_channels, out_channels, heads=4):
        super().__init__()
        self.conv = HeteroConv({
            ('sentence', 'belongs_to', 'paragraph'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('paragraph', 'belongs_to', 'section'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('section', 'belongs_to', 'document'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('sentence', 'similar_to', 'sentence'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
        }, aggr='sum')
    def forward(self, x_dict, edge_index_dict):
        out_dict = self.conv(x_dict, edge_index_dict)
        for node_type in x_dict:
            if node_type not in out_dict:
                out_dict[node_type] = x_dict[node_type]
        return {k: F.elu(v) for k, v in out_dict.items()}

class HierarchicalGAT(nn.Module):
    def __init__(self, hidden_channels=64, heads=4):
        super().__init__()
        self.layer1 = HierarchicalGATLayer(-1, hidden_channels, heads)
        self.layer2 = HierarchicalGATLayer(-1, hidden_channels, heads)
    def forward(self, x_dict, edge_index_dict):
        x_dict = self.layer1(x_dict, edge_index_dict)
        x_dict = self.layer2(x_dict, edge_index_dict)
        return x_dict

class GraphToDecoderBridge(nn.Module):
    def __init__(self, gat_dim=256, decoder_dim=512):
        super().__init__()
        self.projection = nn.Linear(gat_dim, decoder_dim)
        self.norm = nn.LayerNorm(decoder_dim)
    def forward(self, section_reprs):
        projected = self.projection(section_reprs)
        normalized = self.norm(projected)
        return normalized.unsqueeze(0)

class HierarchicalGATSummarizer(nn.Module):
    def __init__(self, t5_model, gat_hidden=64, gat_heads=4, decoder_dim=512):
        super().__init__()
        self.gat_encoder = HierarchicalGAT(hidden_channels=gat_hidden, heads=gat_heads)
        self.bridge = GraphToDecoderBridge(gat_dim=gat_hidden*gat_heads, decoder_dim=decoder_dim)
        self.t5 = t5_model
    def forward(self, x_dict, edge_index_dict, labels=None):
        out_dict = self.gat_encoder(x_dict, edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = self.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=pseudo_output.device)
        return self.t5(encoder_outputs=encoder_outputs, attention_mask=encoder_attention_mask, labels=labels)

# --- Load data, embedder, tokenizer ---
train_data = load_dataset("ccdv/pubmed-summarization", split="train[:1000]")
train_data = train_data.filter(lambda x: len(x["article"].strip()) > 0)

test_data = load_dataset("ccdv/pubmed-summarization", split="test[:50]")
test_data = test_data.filter(lambda x: len(x["article"].strip()) > 0)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
tokenizer = T5Tokenizer.from_pretrained("t5-small")

print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")
print("Setup complete.")

Device: cuda


README.md:   0%|          | 0.00/3.80k [00:00<?, ?B/s]

section/train-00000-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  210MB            

section/train-00000-of-00005.parquet: downloading bytes:           |  0.00B            

section/train-00001-of-00005.parquet: reconstructing file:   0%|          |  0.00B /  208MB            

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Train size: 983, Test size: 50
Setup complete.


In [5]:
model_name = "t5-small"
model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

class SummarizationDataset(TorchDataset):
    def __init__(self, hf_dataset, tokenizer, max_input_len=512, max_target_len=150):
        self.hf_dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
    def __len__(self):
        return len(self.hf_dataset)
    def __getitem__(self, idx):
        article = self.hf_dataset[idx]["article"]
        summary = self.hf_dataset[idx]["abstract"]
        inputs = self.tokenizer("summarize: " + article, max_length=self.max_input_len, truncation=True, padding="max_length", return_tensors="pt")
        labels = tokenize_targets([summary], self.tokenizer, self.max_target_len)
        return {"input_ids": inputs["input_ids"].squeeze(0), "attention_mask": inputs["attention_mask"].squeeze(0), "labels": labels.squeeze(0)}

train_dataset = SummarizationDataset(train_data, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
start_time = time.time()
total_loss = 0
for step, batch in enumerate(train_loader):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()
    if step % 50 == 0:
        print(f"Step {step}/{len(train_loader)} | Loss: {loss.item():.4f}")

print(f"\nBaseline training complete. Avg loss: {total_loss/len(train_loader):.4f}")
torch.save(model.state_dict(), os.path.join(checkpoint_dir, "baseline_t5.pt"))
print("Saved to Drive.")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Step 0/246 | Loss: 6.1688
Step 50/246 | Loss: 3.1741
Step 100/246 | Loss: 2.7250
Step 150/246 | Loss: 3.4306
Step 200/246 | Loss: 2.7587

Baseline training complete. Avg loss: 3.2265


NameError: name 'os' is not defined

In [7]:
import os
checkpoint_dir = "/content/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

torch.save(model.state_dict(), os.path.join(checkpoint_dir, "baseline_t5.pt"))
print("Baseline saved:", os.path.join(checkpoint_dir, "baseline_t5.pt"))

Baseline saved: /content/checkpoints/baseline_t5.pt


In [8]:
fresh_t5 = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
gat_model = HierarchicalGATSummarizer(fresh_t5).to(device)
optimizer = AdamW(gat_model.parameters(), lr=5e-5)
gat_model.train()

num_epochs = 3
train_size = len(train_data)
log_every = 100

start_time = time.time()

for epoch in range(num_epochs):
    total_loss = 0
    for i in range(train_size):
        article = train_data[i]["article"]
        summary = train_data[i]["abstract"]

        doc_graph = build_full_graph(article, embedder).to(device)
        target = tokenize_targets([summary], tokenizer).to(device)

        outputs = gat_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if i % log_every == 0:
            elapsed = time.time() - start_time
            print(f"Epoch {epoch+1} | Step {i}/{train_size} | Loss: {loss.item():.4f} | Elapsed: {elapsed/60:.1f}min")

    avg_loss = total_loss / train_size
    print(f"\n=== Epoch {epoch+1} complete. Avg loss: {avg_loss:.4f} ===\n")

torch.save(gat_model.state_dict(), os.path.join(checkpoint_dir, "hgat_summarizer_final.pt"))
print(f"GAT model saved. Total training time: {(time.time()-start_time)/60:.1f} minutes")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1 | Step 0/983 | Loss: 11.0848 | Elapsed: 0.0min
Epoch 1 | Step 100/983 | Loss: 6.8661 | Elapsed: 0.4min
Epoch 1 | Step 200/983 | Loss: 7.2064 | Elapsed: 0.9min
Epoch 1 | Step 300/983 | Loss: 7.7032 | Elapsed: 1.3min
Epoch 1 | Step 400/983 | Loss: 6.6110 | Elapsed: 1.8min
Epoch 1 | Step 500/983 | Loss: 6.4414 | Elapsed: 2.2min
Epoch 1 | Step 600/983 | Loss: 6.0042 | Elapsed: 2.6min
Epoch 1 | Step 700/983 | Loss: 6.0594 | Elapsed: 3.1min
Epoch 1 | Step 800/983 | Loss: 6.2012 | Elapsed: 3.5min
Epoch 1 | Step 900/983 | Loss: 5.8330 | Elapsed: 3.9min

=== Epoch 1 complete. Avg loss: 6.5801 ===

Epoch 2 | Step 0/983 | Loss: 6.1620 | Elapsed: 4.3min
Epoch 2 | Step 100/983 | Loss: 5.4482 | Elapsed: 4.7min
Epoch 2 | Step 200/983 | Loss: 6.0150 | Elapsed: 5.1min
Epoch 2 | Step 300/983 | Loss: 6.6310 | Elapsed: 5.6min
Epoch 2 | Step 400/983 | Loss: 5.7918 | Elapsed: 6.0min
Epoch 2 | Step 500/983 | Loss: 5.7799 | Elapsed: 6.4min
Epoch 2 | Step 600/983 | Loss: 5.0195 | Elapsed: 6.8min
Epoch 

In [9]:
import evaluate

rouge = evaluate.load("rouge")

def generate_baseline_summary(article, model, tokenizer, max_input_len=512, max_output_len=150):
    inputs = tokenizer("summarize: " + article, max_length=max_input_len, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        summary_ids = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"], max_length=max_output_len, num_beams=4, early_stopping=True)
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

def generate_gat_summary(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs, attention_mask=encoder_attention_mask,
            max_length=max_output_len, num_beams=1, do_sample=False,
            repetition_penalty=2.5, no_repeat_ngram_size=2
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

baseline_preds, gat_preds, references = [], [], []
model.eval()
gat_model.eval()

print("Generating summaries from both models...")
for i in range(len(test_data)):
    article = test_data[i]["article"]
    reference = test_data[i]["abstract"]
    baseline_preds.append(generate_baseline_summary(article, model, tokenizer))
    gat_preds.append(generate_gat_summary(article, gat_model, embedder, tokenizer))
    references.append(reference)
    if i % 10 == 0:
        print(f"  Processed {i}/{len(test_data)}")

print("Computing ROUGE...")
baseline_rouge = rouge.compute(predictions=baseline_preds, references=references)
gat_rouge = rouge.compute(predictions=gat_preds, references=references)

print("\n=== RESULTS TABLE ===")
print(f"{'Metric':<15} {'Baseline':<12} {'Hierarchical GAT':<18}")
print(f"{'ROUGE-1':<15} {baseline_rouge['rouge1']:<12.4f} {gat_rouge['rouge1']:<18.4f}")
print(f"{'ROUGE-2':<15} {baseline_rouge['rouge2']:<12.4f} {gat_rouge['rouge2']:<18.4f}")
print(f"{'ROUGE-L':<15} {baseline_rouge['rougeL']:<12.4f} {gat_rouge['rougeL']:<18.4f}")

# Save predictions immediately as insurance against further session issues
import json
with open(os.path.join(checkpoint_dir, "week6_predictions.json"), "w") as f:
    json.dump({"baseline_preds": baseline_preds, "gat_preds": gat_preds, "references": references}, f)
print("\nPredictions saved as backup.")

Generating summaries from both models...
  Processed 0/50
  Processed 10/50
  Processed 20/50
  Processed 30/50
  Processed 40/50
Computing ROUGE...

=== RESULTS TABLE ===
Metric          Baseline     Hierarchical GAT  
ROUGE-1         0.2280       0.1814            
ROUGE-2         0.0912       0.0252            
ROUGE-L         0.1649       0.1107            

Predictions saved as backup.


In [10]:
import torch

def measure_baseline_efficiency(article, model, tokenizer, max_input_len=512, max_output_len=150):
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize()
    start = time.time()

    inputs = tokenizer("summarize: " + article, max_length=max_input_len, truncation=True, return_tensors="pt").to(device)
    with torch.no_grad():
        summary_ids = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"], max_length=max_output_len, num_beams=4, early_stopping=True)

    torch.cuda.synchronize()
    elapsed = time.time() - start
    peak_mem = torch.cuda.max_memory_allocated(device) / (1024**2)  # MB
    return elapsed, peak_mem

def measure_gat_efficiency(article, model, embedder, tokenizer, max_output_len=150):
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize()
    start = time.time()

    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs, attention_mask=encoder_attention_mask,
            max_length=max_output_len, num_beams=1, do_sample=False,
            repetition_penalty=2.5, no_repeat_ngram_size=2
        )

    torch.cuda.synchronize()
    elapsed = time.time() - start
    peak_mem = torch.cuda.max_memory_allocated(device) / (1024**2)  # MB
    return elapsed, peak_mem

# Run on 20 test documents, track document length too (for length-bucket analysis)
baseline_times, baseline_mems = [], []
gat_times, gat_mems = [], []
doc_lengths = []

for i in range(20):
    article = test_data[i]["article"]
    doc_lengths.append(len(split_sentences(article)))

    b_time, b_mem = measure_baseline_efficiency(article, model, tokenizer)
    baseline_times.append(b_time)
    baseline_mems.append(b_mem)

    g_time, g_mem = measure_gat_efficiency(article, gat_model, embedder, tokenizer)
    gat_times.append(g_time)
    gat_mems.append(g_mem)

print("=== EFFICIENCY RESULTS (avg over 20 docs) ===")
print(f"{'Metric':<25} {'Baseline':<15} {'Hierarchical GAT':<18}")
print(f"{'Avg latency (s/doc)':<25} {sum(baseline_times)/len(baseline_times):<15.3f} {sum(gat_times)/len(gat_times):<18.3f}")
print(f"{'Peak GPU memory (MB)':<25} {sum(baseline_mems)/len(baseline_mems):<15.1f} {sum(gat_mems)/len(gat_mems):<18.1f}")
print(f"{'Avg doc length (sentences)':<25} {sum(doc_lengths)/len(doc_lengths):<15.1f}")

=== EFFICIENCY RESULTS (avg over 20 docs) ===
Metric                    Baseline        Hierarchical GAT  
Avg latency (s/doc)       1.516           1.178             
Peak GPU memory (MB)      1381.0          1374.7            
Avg doc length (sentences) 94.8           


In [11]:
def measure_gat_efficiency_beam(article, model, embedder, tokenizer, max_output_len=150):
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize()
    start = time.time()

    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs, attention_mask=encoder_attention_mask,
            max_length=max_output_len, num_beams=4, early_stopping=True
        )

    torch.cuda.synchronize()
    elapsed = time.time() - start
    peak_mem = torch.cuda.max_memory_allocated(device) / (1024**2)
    return elapsed, peak_mem

gat_times_beam, gat_mems_beam = [], []
for i in range(20):
    article = test_data[i]["article"]
    g_time, g_mem = measure_gat_efficiency_beam(article, gat_model, embedder, tokenizer)
    gat_times_beam.append(g_time)
    gat_mems_beam.append(g_mem)

print("=== EFFICIENCY RESULTS, MATCHED DECODING (beam=4 for both) ===")
print(f"{'Metric':<25} {'Baseline':<15} {'Hierarchical GAT':<18}")
print(f"{'Avg latency (s/doc)':<25} {sum(baseline_times)/len(baseline_times):<15.3f} {sum(gat_times_beam)/len(gat_times_beam):<18.3f}")
print(f"{'Peak GPU memory (MB)':<25} {sum(baseline_mems)/len(baseline_mems):<15.1f} {sum(gat_mems_beam)/len(gat_mems_beam):<18.1f}")

=== EFFICIENCY RESULTS, MATCHED DECODING (beam=4 for both) ===
Metric                    Baseline        Hierarchical GAT  
Avg latency (s/doc)       1.516           1.845             
Peak GPU memory (MB)      1381.0          1375.1            


Ablation A: Remove semantic edges

In [12]:
class HierarchicalGATLayerNoSemantic(nn.Module):
    """Same as HierarchicalGATLayer but WITHOUT the sentence<->sentence semantic edge type."""
    def __init__(self, in_channels, out_channels, heads=4):
        super().__init__()
        self.conv = HeteroConv({
            ('sentence', 'belongs_to', 'paragraph'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('paragraph', 'belongs_to', 'section'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            ('section', 'belongs_to', 'document'): GATConv((-1, -1), out_channels, heads=heads, add_self_loops=False),
            # semantic edge type removed
        }, aggr='sum')
    def forward(self, x_dict, edge_index_dict):
        out_dict = self.conv(x_dict, edge_index_dict)
        for node_type in x_dict:
            if node_type not in out_dict:
                out_dict[node_type] = x_dict[node_type]
        return {k: F.elu(v) for k, v in out_dict.items()}

class HierarchicalGATNoSemantic(nn.Module):
    def __init__(self, hidden_channels=64, heads=4):
        super().__init__()
        self.layer1 = HierarchicalGATLayerNoSemantic(-1, hidden_channels, heads)
        self.layer2 = HierarchicalGATLayerNoSemantic(-1, hidden_channels, heads)
    def forward(self, x_dict, edge_index_dict):
        x_dict = self.layer1(x_dict, edge_index_dict)
        x_dict = self.layer2(x_dict, edge_index_dict)
        return x_dict

class HierarchicalGATSummarizerNoSemantic(nn.Module):
    def __init__(self, t5_model, gat_hidden=64, gat_heads=4, decoder_dim=512):
        super().__init__()
        self.gat_encoder = HierarchicalGATNoSemantic(hidden_channels=gat_hidden, heads=gat_heads)
        self.bridge = GraphToDecoderBridge(gat_dim=gat_hidden*gat_heads, decoder_dim=decoder_dim)
        self.t5 = t5_model
    def forward(self, x_dict, edge_index_dict, labels=None):
        out_dict = self.gat_encoder(x_dict, edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = self.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=pseudo_output.device)
        return self.t5(encoder_outputs=encoder_outputs, attention_mask=encoder_attention_mask, labels=labels)

# Train
fresh_t5_a = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
ablation_a_model = HierarchicalGATSummarizerNoSemantic(fresh_t5_a).to(device)
optimizer_a = AdamW(ablation_a_model.parameters(), lr=5e-5)
ablation_a_model.train()

start_time = time.time()
for epoch in range(3):
    total_loss = 0
    for i in range(len(train_data)):
        article = train_data[i]["article"]
        summary = train_data[i]["abstract"]
        doc_graph = build_full_graph(article, embedder).to(device)
        target = tokenize_targets([summary], tokenizer).to(device)
        outputs = ablation_a_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
        loss = outputs.loss
        optimizer_a.zero_grad()
        loss.backward()
        optimizer_a.step()
        total_loss += loss.item()
        if i % 200 == 0:
            print(f"Epoch {epoch+1} | Step {i} | Loss: {loss.item():.4f} | Elapsed: {(time.time()-start_time)/60:.1f}min")
    print(f"=== Epoch {epoch+1} avg loss: {total_loss/len(train_data):.4f} ===")

torch.save(ablation_a_model.state_dict(), os.path.join(checkpoint_dir, "ablation_a_no_semantic.pt"))
print(f"Ablation A done. Total time: {(time.time()-start_time)/60:.1f} min")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

/tmp/ipykernel_3828/61347043.py:5: UserWarning: There exist node types ({'sentence'}) whose representations do not get updated during message passing as they do not occur as destination type in any edge type. This may lead to unexpected behavior.
  self.conv = HeteroConv({


Epoch 1 | Step 0 | Loss: 11.1557 | Elapsed: 0.0min
Epoch 1 | Step 200 | Loss: 7.1042 | Elapsed: 0.8min
Epoch 1 | Step 400 | Loss: 6.6091 | Elapsed: 1.7min
Epoch 1 | Step 600 | Loss: 5.9055 | Elapsed: 2.5min
Epoch 1 | Step 800 | Loss: 6.3048 | Elapsed: 3.3min
=== Epoch 1 avg loss: 6.5562 ===
Epoch 2 | Step 0 | Loss: 6.1295 | Elapsed: 4.1min
Epoch 2 | Step 200 | Loss: 5.9238 | Elapsed: 5.0min
Epoch 2 | Step 400 | Loss: 5.7064 | Elapsed: 5.8min
Epoch 2 | Step 600 | Loss: 4.7445 | Elapsed: 6.6min
Epoch 2 | Step 800 | Loss: 4.7392 | Elapsed: 7.5min
=== Epoch 2 avg loss: 5.2790 ===
Epoch 3 | Step 0 | Loss: 4.8757 | Elapsed: 8.3min
Epoch 3 | Step 200 | Loss: 4.3169 | Elapsed: 9.1min
Epoch 3 | Step 400 | Loss: 4.7117 | Elapsed: 10.0min
Epoch 3 | Step 600 | Loss: 3.9993 | Elapsed: 10.8min
Epoch 3 | Step 800 | Loss: 4.3847 | Elapsed: 11.7min
=== Epoch 3 avg loss: 4.4085 ===
Ablation A done. Total time: 12.6 min


In [14]:
def generate_gat_summary_generic(article, model, embedder, tokenizer, max_output_len=150):
    doc_graph = build_full_graph(article, embedder).to(device)
    with torch.no_grad():
        out_dict = model.gat_encoder(doc_graph.x_dict, doc_graph.edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = model.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=device)
        summary_ids = model.t5.generate(
            encoder_outputs=encoder_outputs, attention_mask=encoder_attention_mask,
            max_length=max_output_len, num_beams=1, do_sample=False,
            repetition_penalty=2.5, no_repeat_ngram_size=2
        )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

ablation_a_model.eval()
ablation_a_preds = []
for i in range(len(test_data)):
    ablation_a_preds.append(generate_gat_summary_generic(test_data[i]["article"], ablation_a_model, embedder, tokenizer))

ablation_a_rouge = rouge.compute(predictions=ablation_a_preds, references=references)

print("=== ABLATION A: No Semantic Edges ===")
print(f"{'Metric':<15} {'Full GAT':<12} {'No Semantic Edges':<18}")
print(f"{'ROUGE-1':<15} {gat_rouge['rouge1']:<12.4f} {ablation_a_rouge['rouge1']:<18.4f}")
print(f"{'ROUGE-2':<15} {gat_rouge['rouge2']:<12.4f} {ablation_a_rouge['rouge2']:<18.4f}")
print(f"{'ROUGE-L':<15} {gat_rouge['rougeL']:<12.4f} {ablation_a_rouge['rougeL']:<18.4f}")

=== ABLATION A: No Semantic Edges ===
Metric          Full GAT     No Semantic Edges 
ROUGE-1         0.1814       0.1646            
ROUGE-2         0.0252       0.0242            
ROUGE-L         0.1107       0.1037            


Ablation D: GAT vs GCN (RQ2)

In [15]:
from torch_geometric.nn import GCNConv

class HierarchicalGCNLayer(nn.Module):
    """Same structure as HierarchicalGATLayer, but GCNConv instead of GATConv (no attention)."""
    def __init__(self, in_channels, out_channels):
        super().__init__()

        from torch_geometric.nn import SAGEConv
        self.conv = HeteroConv({
            ('sentence', 'belongs_to', 'paragraph'): SAGEConv((-1, -1), out_channels),
            ('paragraph', 'belongs_to', 'section'): SAGEConv((-1, -1), out_channels),
            ('section', 'belongs_to', 'document'): SAGEConv((-1, -1), out_channels),
            ('sentence', 'similar_to', 'sentence'): SAGEConv((-1, -1), out_channels),
        }, aggr='sum')
    def forward(self, x_dict, edge_index_dict):
        out_dict = self.conv(x_dict, edge_index_dict)
        for node_type in x_dict:
            if node_type not in out_dict:
                out_dict[node_type] = x_dict[node_type]
        return {k: F.elu(v) for k, v in out_dict.items()}

class HierarchicalGCN(nn.Module):
    def __init__(self, hidden_channels=256):
        super().__init__()
        self.layer1 = HierarchicalGCNLayer(-1, hidden_channels)
        self.layer2 = HierarchicalGCNLayer(-1, hidden_channels)
    def forward(self, x_dict, edge_index_dict):
        x_dict = self.layer1(x_dict, edge_index_dict)
        x_dict = self.layer2(x_dict, edge_index_dict)
        return x_dict

class HierarchicalGCNSummarizer(nn.Module):
    def __init__(self, t5_model, hidden_channels=256, decoder_dim=512):
        super().__init__()
        self.gat_encoder = HierarchicalGCN(hidden_channels=hidden_channels)
        self.bridge = GraphToDecoderBridge(gat_dim=hidden_channels, decoder_dim=decoder_dim)
        self.t5 = t5_model
    def forward(self, x_dict, edge_index_dict, labels=None):
        out_dict = self.gat_encoder(x_dict, edge_index_dict)
        section_reprs = out_dict['section']
        pseudo_output = self.bridge(section_reprs)
        num_sections = section_reprs.shape[0]
        encoder_outputs = BaseModelOutput(last_hidden_state=pseudo_output)
        encoder_attention_mask = torch.ones(1, num_sections, dtype=torch.long, device=pseudo_output.device)
        return self.t5(encoder_outputs=encoder_outputs, attention_mask=encoder_attention_mask, labels=labels)

# Train
fresh_t5_d = T5ForConditionalGeneration.from_pretrained("t5-small").to(device)
ablation_d_model = HierarchicalGCNSummarizer(fresh_t5_d).to(device)
optimizer_d = AdamW(ablation_d_model.parameters(), lr=5e-5)
ablation_d_model.train()

start_time = time.time()
for epoch in range(3):
    total_loss = 0
    for i in range(len(train_data)):
        article = train_data[i]["article"]
        summary = train_data[i]["abstract"]
        doc_graph = build_full_graph(article, embedder).to(device)
        target = tokenize_targets([summary], tokenizer).to(device)
        outputs = ablation_d_model(doc_graph.x_dict, doc_graph.edge_index_dict, labels=target)
        loss = outputs.loss
        optimizer_d.zero_grad()
        loss.backward()
        optimizer_d.step()
        total_loss += loss.item()
        if i % 200 == 0:
            print(f"Epoch {epoch+1} | Step {i} | Loss: {loss.item():.4f} | Elapsed: {(time.time()-start_time)/60:.1f}min")
    print(f"=== Epoch {epoch+1} avg loss: {total_loss/len(train_data):.4f} ===")

torch.save(ablation_d_model.state_dict(), os.path.join(checkpoint_dir, "ablation_d_gcn.pt"))
print(f"Ablation D done. Total time: {(time.time()-start_time)/60:.1f} min")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Epoch 1 | Step 0 | Loss: 10.6154 | Elapsed: 0.0min
Epoch 1 | Step 200 | Loss: 8.9655 | Elapsed: 0.8min
Epoch 1 | Step 400 | Loss: 8.8135 | Elapsed: 1.7min
Epoch 1 | Step 600 | Loss: 6.7586 | Elapsed: 2.5min
Epoch 1 | Step 800 | Loss: 7.1705 | Elapsed: 3.3min
=== Epoch 1 avg loss: 7.6768 ===
Epoch 2 | Step 0 | Loss: 7.3827 | Elapsed: 4.1min
Epoch 2 | Step 200 | Loss: 7.1414 | Elapsed: 4.9min
Epoch 2 | Step 400 | Loss: 7.0291 | Elapsed: 5.8min
Epoch 2 | Step 600 | Loss: 6.2777 | Elapsed: 6.6min
Epoch 2 | Step 800 | Loss: 6.9236 | Elapsed: 7.4min
=== Epoch 2 avg loss: 6.6734 ===
Epoch 3 | Step 0 | Loss: 6.9608 | Elapsed: 8.2min
Epoch 3 | Step 200 | Loss: 6.8071 | Elapsed: 9.0min
Epoch 3 | Step 400 | Loss: 6.6298 | Elapsed: 9.8min
Epoch 3 | Step 600 | Loss: 6.0787 | Elapsed: 10.7min
Epoch 3 | Step 800 | Loss: 6.8449 | Elapsed: 11.5min
=== Epoch 3 avg loss: 6.5266 ===
Ablation D done. Total time: 12.4 min


In [16]:
ablation_d_model.eval()
ablation_d_preds = []
for i in range(len(test_data)):
    ablation_d_preds.append(generate_gat_summary_generic(test_data[i]["article"], ablation_d_model, embedder, tokenizer))

ablation_d_rouge = rouge.compute(predictions=ablation_d_preds, references=references)

print("=== ABLATION D: GAT vs GCN (SAGEConv) ===")
print(f"{'Metric':<15} {'Full GAT':<12} {'GCN (SAGEConv)':<18}")
print(f"{'ROUGE-1':<15} {gat_rouge['rouge1']:<12.4f} {ablation_d_rouge['rouge1']:<18.4f}")
print(f"{'ROUGE-2':<15} {gat_rouge['rouge2']:<12.4f} {ablation_d_rouge['rouge2']:<18.4f}")
print(f"{'ROUGE-L':<15} {gat_rouge['rougeL']:<12.4f} {ablation_d_rouge['rougeL']:<18.4f}")

# Save all predictions as final backup
with open(os.path.join(checkpoint_dir, "week6_full_results.json"), "w") as f:
    json.dump({
        "baseline_preds": baseline_preds, "gat_preds": gat_preds,
        "ablation_a_preds": ablation_a_preds, "ablation_d_preds": ablation_d_preds,
        "references": references,
        "rouge_scores": {
            "baseline": baseline_rouge, "gat": gat_rouge,
            "ablation_a_no_semantic": ablation_a_rouge, "ablation_d_gcn": ablation_d_rouge
        }
    }, f)
print("\nAll Week 6 results saved.")

=== ABLATION D: GAT vs GCN (SAGEConv) ===
Metric          Full GAT     GCN (SAGEConv)    
ROUGE-1         0.1814       0.0812            
ROUGE-2         0.0252       0.0030            
ROUGE-L         0.1107       0.0642            

All Week 6 results saved.
